> **English summary.** This notebook is the full analysis behind my Master's thesis: a quarterly urbanisation index for Nouakchott built from Sentinel-1 radar on Google Earth Engine (2016 – 2026 Q2), validated against ANSADE's Construction Cost Index with cointegration and a VECM. Commentary is in French; the project overview, figures and results are in English in the repository's `README.md`.
>
> **To run:** authenticate Earth Engine (`earthengine authenticate`), set `GEE_PROJECT` to your Google Cloud project, and run from the `notebooks/` folder. Outputs are written to `../outputs/`.


# 🛰️ Notebook 1b — Cartographie SAR de l'Urbanisation de Nouakchott
## Ground Truth · SAR Backscatter · Indice Urbain · Analyse Économique

**Objectif :** Cartographier l'expansion urbaine de Nouakchott (2016 – 2026) via Sentinel-1 SAR,  
construire un indice composite par moughataa, et analyser les corrélations économiques.

---

### 📋 Plan du notebook

| Partie | Description |
|--------|-------------|
| **1** | Setup, shapefile, points terrain |
| **2** | Analyse spectrale Sentinel-2 — indices optiques |
| **3** | Détection de changement optique — limites |
| **4** | Disponibilité & validation SAR Sentinel-1 |
| **5** | Classification urbaine SAR (seuil -14 dB) |
| **6** | Visualisations SAR multi-années (2016→2024) |
| **7** | Série temporelle annuelle par moughataa |
| **8** | Indice composite & indice synthétique ville |
| **9** | Données économiques & corrélations |
| **10** | Tests économétriques de robustesse |

---
> **Sources :** Sentinel-1 GRD · Sentinel-2 SR · ANSADE/ICC · World Bank WDI  
> **Plateforme :** Google Earth Engine (Python API)  
> **Zone :** Ville de Nouakchott — 9 moughataat · **Période :** 2016 – 2026 T2

---
## 🗺️ PARTIE 1 — Setup & Points de Vérité Terrain

Les 6 points ont été sélectionnés sur **Google Earth** pour représenter des classes  
de surface connues avec certitude : urbain dense, sable pur, sebkha/lagon, route.

> Ces points servent d'ancre pour toute l'analyse : calibration des seuils SAR,  
> validation des classifications et interprétation des cartes.

In [ ]:
# ── CELL 1 — Imports, GEE Init, Shapefile, Points Terrain ──────────────────
import ee
import geopandas as gpd
import shapely.ops
import json as _json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
import requests
from io import BytesIO
from PIL import Image
import os
os.makedirs('../outputs', exist_ok=True)

ee.Initialize(project=os.environ.get('GEE_PROJECT'))
print('✓ GEE initialized')

# ── Shapefile Nouakchott ──────────────────────────────────────
gdf_all = gpd.read_file('../data/boundaries/nouakchott_moughataa.geojson')
if gdf_all.crs.to_epsg() != 4326:
    gdf_all = gdf_all.to_crs(epsg=4326)
gdf       = gdf_all[gdf_all['ADM1_EN'] == 'Nouakchott'].copy().reset_index(drop=True)
bounds    = gdf.total_bounds
union     = shapely.ops.unary_union(gdf.geometry)
gj        = _json.loads(gpd.GeoSeries([union], crs='EPSG:4326').to_json())
region_ee = ee.Geometry(gj['features'][0]['geometry'])
print(f'✓ Shapefile: {len(gdf)} moughataat chargées')

# ── Ground truth points ───────────────────────────────────────
GROUND_TRUTH = {
    'Urban 1'   : {'lat':  18.093214, 'lon': -15.980358, 'type': 'urban'},
    'Urban 2'   : {'lat':  18.081911, 'lon': -15.977133, 'type': 'urban'},
    'Sand 1'    : {'lat':  18.026425, 'lon': -16.010217, 'type': 'sand'},
    'Sand 2'    : {'lat':  18.198997, 'lon': -15.957258, 'type': 'sand'},
    'Lagoon'    : {'lat':  18.043989, 'lon': -15.980083, 'type': 'water'},
    'Major Road': {'lat':  18.087844, 'lon': -15.980406, 'type': 'road'},
}

TYPE_COLORS = {
    'urban': '#d62728', 'sand': '#f4a261',
    'water': '#1f77b4', 'road': '#7f7f7f',
}

ZONE_ORDER = ['Arafat','Dar Naim','Toujounine',
              'Ksar','Tevragh Zein','Sebkha',
              'Teyarett','El Mina','Riyadh']

ZONE_COLORS = {
    'Arafat':'#e63946',       'Dar Naim':'#f4722b',
    'El Mina':'#f4a261',      'Ksar':'#2a9d8f',
    'Tevragh Zein':'#1d7a72', 'Sebkha':'#52b788',
    'Toujounine':'#3a86ff',   'Teyarett':'#8338ec',
    'Riyadh':'#6c757d',
}

print(f'\n{"Point":12s} {"Type":10s} {"Lat":12s} {"Lon":12s}')
print('-' * 50)
for name, info in GROUND_TRUTH.items():
    print(f"{name:12s} {info['type']:10s} {info['lat']:12.6f} {info['lon']:12.6f}")
print('\n✓ Setup complet')

In [ ]:
# ── CELL 1b — Configuration temporelle (source unique de vérité) ────────────
from datetime import date

YEAR_START    = 2016
ANNUAL_END    = 2025            # dernière année civile COMPLÈTE
MONTHLY_END   = '2026-06'       # dernier mois complet (juin 2026)
S1_FILTER_END = '2026-06-30'    # borne haute des filtres GEE Sentinel-1

YEARS         = list(range(YEAR_START, ANNUAL_END + 1))   # annuel : 2016..2025
YEARS_MONTHLY = list(range(YEAR_START, 2027))             # mensuel : inclut 2026 partiel
MONTHLY_LAST  = date(2026, 6, 1)                          # garde-fou d'extraction

# Composites SAR
BASE_START,   BASE_END   = '2016-01-01', '2018-12-31'     # début de série
RECENT_START, RECENT_END = '2022-01-01', S1_FILTER_END    # état récent

# Années jalons pour cartes d'évolution (5 années → grille 2×5 de la CELL 12)
MILESTONE_YEARS = [2016, 2018, 2021, 2023, 2025]

print(f'✓ Config | annuel {YEAR_START}-{ANNUAL_END} · mensuel → {MONTHLY_END} · trimestriel → 2026-Q2')

---
## 📡 PARTIE 2 — Analyse Spectrale Sentinel-2 (Indices Optiques)

### Pourquoi tester l'optique d'abord ?
Les indices NDBI, NDWI, BAEI sont la méthode standard pour détecter le bâti.  
On les évalue sur nos points terrain pour **quantifier leurs limites en contexte saharien**.

> ⚠️ Résultat attendu : le sable et le béton ont des signatures SWIR très proches  
> → séparation impossible par seuillage simple.

In [ ]:
# ── CELL 2 — Composite Sentinel-2 (Oct 2023 → Avr 2024) ────────────────────
# Saison sèche : maximum de pixels propres, minimum de poussière atmosphérique

lats = [v['lat'] for v in GROUND_TRUTH.values()]
lons = [v['lon'] for v in GROUND_TRUTH.values()]
region_s2 = ee.Geometry.Rectangle([
    min(lons)-0.05, min(lats)-0.05, max(lons)+0.05, max(lats)+0.05
])

def mask_clouds_s2(img):
    scl     = img.select('SCL')
    ok      = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    dust_ok = img.select('B1').toFloat().divide(img.select('B2').toFloat().add(1e-6)).lt(0.90)
    return img.updateMask(ok.And(dust_ok)).divide(10000).copyProperties(img, ['system:time_start'])

col_s2    = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
               .filterDate('2023-10-01','2024-04-01').filterBounds(region_s2)
               .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)).map(mask_clouds_s2))
n_s2      = col_s2.size().getInfo()
comp_s2   = col_s2.median()
# Aussi composite pour masque eau
comp_after_s2 = comp_s2   # alias utilisé pour NDWI dans CELL 11

print(f'✓ Composite S2 : {n_s2} images (Oct 2023 → Avr 2024)')

In [ ]:
# ── CELL 3 — Extraction Bandes & Calcul Indices aux Points Terrain ─────────
BANDS      = ['B2','B3','B4','B8','B11','B12']
BAND_NAMES = {'B2':'Blue(490)','B3':'Green(560)','B4':'Red(665)',
              'B8':'NIR(842)','B11':'SWIR1(1610)','B12':'SWIR2(2190)'}

records = []
for name, info in GROUND_TRUTH.items():
    pt  = ee.Geometry.Point([info['lon'], info['lat']]).buffer(30)
    val = comp_s2.select(BANDS).reduceRegion(ee.Reducer.mean(), pt, 10, maxPixels=1e6).getInfo()
    row = {'point': name, 'type': info['type']}; row.update(val)
    records.append(row)

df_s2 = pd.DataFrame(records).set_index('point')
for b in BANDS:
    df_s2[b] = df_s2[b].round(4)

eps = 1e-10
def compute_indices(r):
    b2,b3,b4,b8,b11,b12 = r['B2'],r['B3'],r['B4'],r['B8'],r['B11'],r['B12']
    return pd.Series({
        'NDVI'       : round((b8-b4)/(b8+b4+eps),4),
        'NDBI'       : round((b11-b8)/(b11+b8+eps),4),
        'NDWI'       : round((b3-b8)/(b3+b8+eps),4),
        'BU'         : round(((b11-b8)/(b11+b8+eps))-((b8-b4)/(b8+b4+eps)),4),
        'BAEI'       : round((b4+0.3)/(b3+b11+eps),4),
        'SWIR2/SWIR1': round(b12/(b11+eps),4),
        'NDBAI'      : round((b11-b3)/(b11+b3+eps),4),
    })

idx_df = df_s2.apply(compute_indices, axis=1)
idx_df.insert(0, 'type', df_s2['type'])
print('Valeurs des indices aux points terrain :')
print(idx_df.to_string())

urban_ndbi = idx_df[idx_df.type=='urban']['NDBI'].mean()
sand_ndbi  = idx_df[idx_df.type=='sand']['NDBI'].mean()
print(f'\nNDBI Urbain={urban_ndbi:.3f} | Sable={sand_ndbi:.3f} | Écart={abs(urban_ndbi-sand_ndbi):.3f}')
print('→ Écart ~0.05 — trop faible pour seuillage fiable en contexte saharien')

In [ ]:
# ── CELL 4 — Profils Spectraux ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Profils Spectraux aux Points Terrain — Nouakchott Oct2023-Avr2024\n'
             'Gauche: Réflectance brute  |  Droite: Indices dérivés',
             fontsize=12, fontweight='bold')

band_labels  = [BAND_NAMES[b] for b in BANDS]
index_labels = ['NDVI','NDBI','NDWI','BU','BAEI','SWIR2/SWIR1','NDBAI']

for ax, (data, labels, ylabel, title) in zip(axes, [
    (df_s2,  band_labels,  'Réflectance (0-1)',  'Réflectance Brute par Bande'),
    (idx_df, index_labels, "Valeur de l'indice", 'Indices Spectraux Dérivés'),
]):
    for name, row in data.iterrows():
        color  = TYPE_COLORS[row['type']]
        vals   = [row[l] if l in row else row[l] for l in (BANDS if 'NDVI' not in labels else index_labels)]
        ls     = '-' if '1' in name or name in ['Lagoon','Major Road'] else '--'
        ax.plot(range(len(labels)), vals, color=color, lw=2, ls=ls,
                marker='o' if ax==axes[0] else 's', ms=6, label=f"{name} ({row['type']})")
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel(ylabel); ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=8); ax.grid(alpha=0.3, ls=':')
    ax.axhline(0, color='black', lw=0.8, alpha=0.5)

plt.tight_layout()
plt.savefig('../outputs/gt_01_spectral_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: gt_01_spectral_profiles.png')

In [ ]:
# ── CELL 4b — Carte Satellite + NDBI Optique sur Nouakchott ─────────────────
# Visualisation de POURQUOI l'optique échoue : carte RGB + NDBI côte à côte
minx, miny, maxx, maxy = bounds

def get_thumb(image, region, bands, vmin, vmax, dims=700, gamma=1.0):
    url = image.getThumbURL({'bands':bands,'min':vmin,'max':vmax,'gamma':gamma,
                              'region':region,'dimensions':dims,'format':'png'})
    return Image.open(BytesIO(requests.get(url, timeout=90).content))

def get_colored_thumb(image, band, region, palette, vmin, vmax, dims=700):
    url = image.select(band).getThumbURL({'min':vmin,'max':vmax,'palette':palette,
                                           'region':region,'dimensions':dims,'format':'png'})
    return Image.open(BytesIO(requests.get(url, timeout=90).content))

def annotate_map(ax, gdf, bounds, buf=0.02, lc='white'):
    ax.set_xlim(bounds[0]-buf, bounds[2]+buf); ax.set_ylim(bounds[1]-buf, bounds[3]+buf)
    gdf.boundary.plot(ax=ax, color='yellow', linewidth=1.2, alpha=0.9)
    for _, row in gdf.iterrows():
        ax.annotate(row['ADM2_EN'], (row.geometry.centroid.x, row.geometry.centroid.y),
                    fontsize=7, color=lc, ha='center', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2',facecolor='black',alpha=0.5,linewidth=0))
    ax.set_xticks([]); ax.set_yticks([])

def plot_gt(ax):
    for name, info in GROUND_TRUTH.items():
        ax.plot(info['lon'], info['lat'], 'o', color=TYPE_COLORS[info['type']],
                ms=8, markeredgecolor='white', markeredgewidth=1.2)

PALETTE_VV    = ['000000','404040','808080','c0c0c0','ffffff']
PALETTE_URBAN = ['1a1a2e','e63946']
PALETTE_DIFF  = ['1d3557','457b9d','f0f4f8','f4a261','e63946']

print('✓ Fonctions définies')
print('Téléchargement thumbnails Sentinel-2...')

# RGB haute qualité
rgb_thumb = get_thumb(comp_s2, region_ee, ['B4','B3','B2'], 0.02, 0.30, dims=800, gamma=1.4)

# NDBI = (SWIR1 - NIR) / (SWIR1 + NIR)
eps   = 1e-10
ndbi  = (comp_s2.select('B11').subtract(comp_s2.select('B8'))
         .divide(comp_s2.select('B11').add(comp_s2.select('B8')).add(eps))
         .rename('NDBI'))

# NDWI pour référence eau
ndwi_vis = (comp_s2.select('B3').subtract(comp_s2.select('B8'))
            .divide(comp_s2.select('B3').add(comp_s2.select('B8')).add(eps))
            .rename('NDWI'))

# Seuil NDBI "idéal" en contexte tempéré = +0.1 → ici on voit pourquoi il échoue
ndbi_thumb      = get_colored_thumb(ndbi, 'NDBI', region_ee,
                    ['2166ac','4393c3','92c5de','fddbc7','f4a582','d6604d','b2182b'], -0.2, 0.4)
ndbi_mask_thumb = get_colored_thumb(ndbi.gt(0.1).rename('urban_ndbi'), 'urban_ndbi', region_ee,
                    ['f0f0f0','d62728'], 0, 1)
ndwi_thumb      = get_colored_thumb(ndwi_vis, 'NDWI', region_ee,
                    ['d7191c','fdae61','ffffbf','abd9e9','2c7bb6'], -0.4, 0.2)

print('✓ Thumbnails prêts')

fig, axes = plt.subplots(1, 4, figsize=(24, 9))
fig.suptitle('Analyse Spectrale Optique — Sentinel-2 Nouakchott Oct2023-Avr2024\n'
             'Pourquoi NDBI échoue en contexte saharien : sable et béton ont le même NDBI',
             fontsize=13, fontweight='bold')

for ax, (img, title) in zip(axes, [
    (rgb_thumb,       'RGB Sentinel-2\n(référence visuelle)'),
    (ndbi_thumb,      'NDBI = (SWIR1-NIR)/(SWIR1+NIR)\nRouge=haut | Bleu=bas'),
    (ndbi_mask_thumb, 'Masque NDBI > 0.1\n(seuil classique "zones bâties")'),
    (ndwi_thumb,      'NDWI = (Green-NIR)/(Green+NIR)\nBleu=eau | Rouge=sec'),
]):
    ax.imshow(img, extent=[minx, maxx, miny, maxy], origin='upper')
    ax.set_aspect('equal')
    annotate_map(ax, gdf, bounds)
    plot_gt(ax)
    ax.set_title(title, fontsize=9, fontweight='bold')

# Annotation explicative sur la carte NDBI
axes[1].text(0.02, 0.04,
    'Urban1: NDBI=+0.11\nSable:  NDBI=+0.15\n→ Sable > Urbain !\nSéparation impossible',
    transform=axes[1].transAxes, fontsize=8.5, color='white',
    bbox=dict(facecolor='black', alpha=0.7, boxstyle='round'))

# Annotation sur le masque
axes[2].text(0.02, 0.04,
    'Faux positifs massifs\nsur zones sableuses\n(sable classé "bâti")',
    transform=axes[2].transAxes, fontsize=8.5, color='white',
    bbox=dict(facecolor='black', alpha=0.7, boxstyle='round'))

# Légende points terrain
legend_els = [Line2D([0],[0], marker='o', color='w',
                     markerfacecolor=TYPE_COLORS[t], markersize=8, label=t.capitalize())
              for t in ['urban','sand','water','road']]
axes[0].legend(handles=legend_els, loc='lower left', fontsize=8, framealpha=0.8)

plt.tight_layout()
plt.savefig('../outputs/gt_00b_optical_ndbi_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: gt_00b_optical_ndbi_map.png')
print()
print('Valeurs NDBI aux points terrain :')
for name, row in idx_df.iterrows():
    flag = '⚠️ sable > urbain!' if row['type']=='sand' and row['NDBI'] > 0.10 else ''
    print(f'  {name:12s} ({row["type"]:6s}): NDBI = {row["NDBI"]:+.4f}  {flag}')

---
## 🔄 PARTIE 3 — Détection de Changement Optique (et ses Limites)

### Hypothèse testée
Si la valeur absolue échoue (sable ≈ béton), peut-on détecter le **changement** entre 2015/16 et 2023/24 ?

**Logique :** Sable→Béton = ΔSWIR1 négatif (0.44 → 0.35 = -0.09)

### Trois raisons pour lesquelles ça échoue à Nouakchott
1. **Baseline trop courte** : seulement ~8 images S2 disponibles en 2015/16 → médiane instable
2. **Décalage atmosphérique global** : médiane de ΔSWIR1 = -0.020 sur TOUS les pixels (même le sable)
3. **Faux positifs végétation** : ΔSWIR1 Sand1 = -0.040 (confirmé sable pur sur Google Earth)

In [ ]:
# ── CELL 5 — Composites Temporels Avant/Après + Baseline Stable ────────────
def build_s2_comp(d1, d2, region):
    def mask(img):
        scl     = img.select('SCL')
        ok      = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
        dust_ok = img.select('B1').toFloat().divide(img.select('B2').toFloat().add(1e-6)).lt(0.90)
        return img.updateMask(ok.And(dust_ok)).divide(10000).copyProperties(img,['system:time_start'])
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
             .filterDate(d1,d2).filterBounds(region).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',30)).map(mask))
    n   = col.size().getInfo()
    print(f'  {d1} → {d2}: {n} images')
    return col.median(), n

print('Composites S2 :')
comp_before,   n1 = build_s2_comp('2015-10-01','2016-04-01', region_ee)   # court
comp_after_ref,n2 = build_s2_comp('2023-10-01','2024-04-01', region_ee)   # récent
comp_baseline, n3 = build_s2_comp('2015-10-01','2018-04-01', region_ee)   # stable 3 ans
print(f'  → Baseline courte: {n1} images  |  Baseline stable: {n3} images  |  Récent: {n2} images')
print(f'  ⚠️  {n1} images en 2015/16 (S2 lancé juin 2015) = médiane instable')

In [ ]:
# ── CELL 6 — Validation Signal Changement + Distribution ΔSWIR1 ─────────────
eps = 1e-10
delta_swir1_s = comp_after_ref.select('B11').subtract(comp_baseline.select('B11')).rename('delta')
ndvi_base     = (comp_baseline.select('B8').subtract(comp_baseline.select('B4'))
                 .divide(comp_baseline.select('B8').add(comp_baseline.select('B4')).add(eps)))
ndvi_after_r  = (comp_after_ref.select('B8').subtract(comp_after_ref.select('B4'))
                 .divide(comp_after_ref.select('B8').add(comp_after_ref.select('B4')).add(eps)))
delta_ndvi_s  = ndvi_after_r.subtract(ndvi_base).rename('delta_ndvi')

# Validation aux points
print('ΔSWIR1 aux points terrain (baseline stable 2015-2018 vs 2023/24):')
print(f'{"Point":12s} {"Type":6s} {"B11_base":>10s} {"B11_après":>10s} {"ΔSWIR1":>9s}  Résultat')
print('-' * 70)
for name, info in GROUND_TRUTH.items():
    pt  = ee.Geometry.Point([info['lon'],info['lat']]).buffer(30)
    vb  = comp_baseline.select('B11').reduceRegion(ee.Reducer.mean(),pt,10,maxPixels=1e6).getInfo()
    va  = comp_after_ref.select('B11').reduceRegion(ee.Reducer.mean(),pt,10,maxPixels=1e6).getInfo()
    bef = round(float(vb['B11']),4); aft = round(float(va['B11']),4); d = round(aft-bef,4)
    flag = '⚠️ faux positif' if info['type']=='sand' and d<-0.02 else ('✓' if info['type']=='urban' and d<-0.02 else 'stable')
    print(f'{name:12s} {info["type"]:6s} {bef:>10.4f} {aft:>10.4f} {d:>9.4f}  {flag}')

# Distribution sur 500 pixels aléatoires
sample = (delta_swir1_s.addBands(delta_ndvi_s)
          .sample(region=region_ee, scale=30, numPixels=500, seed=42, geometries=False).getInfo())
df_sample = pd.DataFrame([f['properties'] for f in sample['features']])
print(f'\nDistribution ΔSWIR1 (500 pixels aléatoires) :')
print(f'  Médiane : {df_sample["delta"].median():+.4f}  ← TOUS les pixels ont baissé')
print(f'  Cause   : décalage atmosphérique/humidité entre les deux saisons')
print(f'  Conclusion : signal construction (~0.03) < bruit (~0.02-0.04) → méthode inexploitable')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribution ΔSWIR1 — Diagnostic Détection de Changement Optique\n'
             '500 pixels aléatoires — Nouakchott', fontsize=12, fontweight='bold')
ax = axes[0]
ax.hist(df_sample['delta'], bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(-0.04, color='red',   lw=2, ls='--', label='Seuil -0.04')
ax.axvline(df_sample['delta'].median(), color='black', lw=2, label=f'Médiane={df_sample["delta"].median():+.3f}')
ax.set_xlabel('ΔSWIR1'); ax.set_ylabel('Pixels'); ax.set_title('Distribution complète', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3, ls=':')
ax = axes[1]
ax.scatter(df_sample['delta'], df_sample['delta_ndvi'], alpha=0.4, s=15, c='steelblue')
ax.axvline(-0.04, color='red',   lw=1.5, ls='--', label='Seuil SWIR1')
ax.axhline(-0.03, color='green', lw=1.5, ls='--', label='Seuil NDVI')
ax.set_xlabel('ΔSWIR1'); ax.set_ylabel('ΔNDVI')
ax.set_title('ΔSWIR1 vs ΔNDVI — Séparation Construction/Végétation', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.3, ls=':')
plt.tight_layout()
plt.savefig('../outputs/gt_03_delta_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n❌ CONCLUSION : Changement optique SWIR1 inexploitable → passage au SAR obligatoire')

In [ ]:
# ── CELL 6b — Carte Visualisation Changement Optique ────────────────────────
# Montre visuellement le résultat du changement détection et ses faux positifs

print('Téléchargement thumbnails changement optique...')

eps = 1e-10

# Delta SWIR1 (baseline stable 2015-2018 vs 2023/24)
delta_swir1_viz = (comp_after_ref.select('B11')
                   .subtract(comp_baseline.select('B11'))
                   .rename('delta_swir1'))

# Score de construction multi-bandes (version initiale)
ndvi_base_v  = (comp_baseline.select('B8').subtract(comp_baseline.select('B4'))
                .divide(comp_baseline.select('B8').add(comp_baseline.select('B4')).add(eps)))
ndvi_after_v = (comp_after_ref.select('B8').subtract(comp_after_ref.select('B4'))
                .divide(comp_after_ref.select('B8').add(comp_after_ref.select('B4')).add(eps)))
delta_ndvi_v = ndvi_after_v.subtract(ndvi_base_v)

# Masque "construction" avec seuil -0.04 (trop de faux positifs)
is_constr_loose = delta_swir1_viz.lt(-0.04).And(delta_ndvi_v.gt(-0.03)).rename('construction')
# Masque "construction" avec seuil -0.06 (encore des faux positifs)
is_constr_tight = delta_swir1_viz.lt(-0.06).And(delta_ndvi_v.gt(-0.03)).rename('construction')

# Thumbnails
rgb_before_thumb = get_thumb(comp_baseline,    region_ee, ['B4','B3','B2'], 0.02, 0.30, gamma=1.4)
rgb_after_thumb  = get_thumb(comp_after_ref,   region_ee, ['B4','B3','B2'], 0.02, 0.30, gamma=1.4)
delta_thumb      = get_colored_thumb(delta_swir1_viz, 'delta_swir1', region_ee,
                       ['d7191c','f7f7f7','2c7bb6'], -0.08, 0.08)
loose_thumb      = get_colored_thumb(is_constr_loose, 'construction', region_ee,
                       ['cccccc','e63946'], 0, 1)
tight_thumb      = get_colored_thumb(is_constr_tight, 'construction', region_ee,
                       ['cccccc','e63946'], 0, 1)

print('✓ OK')

fig, axes = plt.subplots(1, 5, figsize=(30, 9))
fig.suptitle('Détection de Changement Optique (SWIR1) — Résultats et Faux Positifs\n'
             'Baseline stable 2015-2018 vs 2023/24  |   Méthode rejetée → passage au SAR',
             fontsize=13, fontweight='bold')

for ax, (img, title) in zip(axes, [
    (rgb_before_thumb, 'RGB Baseline\n2015-2018'),
    (rgb_after_thumb,  'RGB Récent\n2023/24'),
    (delta_thumb,      'ΔSWIR1 = Après − Avant\nRouge=baisse | Bleu=hausse\n(baisse = construction potentielle)'),
    (loose_thumb,      'Signal Construction\nSeuil ΔSWIR1 < -0.04\n⚠️ Trop de faux positifs'),
    (tight_thumb,      'Signal Construction\nSeuil ΔSWIR1 < -0.06\n⚠️ Signal trop faible'),
]):
    ax.imshow(img, extent=[minx, maxx, miny, maxy], origin='upper')
    ax.set_aspect('equal')
    annotate_map(ax, gdf, bounds)
    plot_gt(ax)
    ax.set_title(title, fontsize=8.5, fontweight='bold')

# Annoter Sand 1 sur les cartes de changement pour montrer le faux positif
for ax in axes[3:]:
    info = GROUND_TRUTH['Sand 1']
    ax.annotate('Sand 1\n(faux positif!)',
                (info['lon'], info['lat']),
                xytext=(15, 20), textcoords='offset points',
                fontsize=8, color='yellow', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='yellow', lw=1.5),
                bbox=dict(facecolor='black', alpha=0.6, boxstyle='round'))

axes[2].text(0.02, 0.04,
    f'Médiane globale:\nΔSWIR1 = -0.020\n(TOUS les pixels\nont baissé)',
    transform=axes[2].transAxes, fontsize=8, color='white',
    bbox=dict(facecolor='black', alpha=0.7, boxstyle='round'))

plt.tight_layout()
plt.savefig('../outputs/gt_02b_optical_change_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: gt_02b_optical_change_map.png')
print()
print('❌ CONCLUSION VISUELLE :')
print('   - Carte 4 (seuil -0.04) : le sable du désert entier est classé "construction"')
print('   - Carte 5 (seuil -0.06) : signal quasi inexistant, rien de détectable')
print('   → Impossible de distinguer construction réelle du bruit dans les deux cas')
print('   → RAISON : décalage atmosphérique global > signal construction')

---
## 📻 PARTIE 4 — Disponibilité & Validation SAR Sentinel-1

### Pourquoi le SAR fonctionne là où l'optique échoue

| Surface | σ⁰ VV typique | Physique |
|---------|:------------:|---------|
| Sable lisse | -20 à -22 dB | Réflexion spéculaire — radar rebondit loin |
| Sebkha/eau | -15 à -18 dB | Idem |
| Route bitumée | -7 à -9 dB | Rugosité planaire |
| Bâti dense | -7 à -10 dB | Double-bounce mur/sol → signal fort |

L'écart **urbain/sable ≈ 10-13 dB** est structurel et stable dans le temps.  
Il ne dépend pas de l'humidité du sol ni des conditions atmosphériques.

In [ ]:
# ── CELL 7 — Disponibilité Sentinel-1 (orbite ASCENDANTE uniquement) ────────
# Note: orbite DESCENDANTE = 0 images sur Nouakchott (à RE-vérifier — bloc en bas)
# Seule l'orbite ASCENDANTE couvre cette région de Mauritanie

s1_col = (ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterBounds(region_ee)
            .filterDate('2015-01-01', S1_FILTER_END)
            .filter(ee.Filter.eq('instrumentMode','IW'))
            .filter(ee.Filter.eq('orbitProperties_pass','ASCENDING'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation','VV')))

print('Couverture Sentinel-1 ASCENDING — Nouakchott :')
print(f'{"Année":6s}  {"Images":>6s}  Couverture')
print('-' * 50)
for year in range(2015, 2027):
    d2  = S1_FILTER_END if year == 2026 else f'{year}-12-31'
    n   = s1_col.filterDate(f'{year}-01-01', d2).size().getInfo()
    tag = '  (partiel → juin)' if year == 2026 else ''
    print(f'{year}    {n:>4d}  {"█"*n}{tag}')
print(f'\nTotal 2015 → {MONTHLY_END} : {s1_col.size().getInfo()} images')
print('✓ ~24-31 images/an depuis 2016 → composites annuels très stables')

# ── (À lancer une fois) Vérifier l'orbite DESCENDANTE avant tout plan robustesse
# desc = (ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(region_ee)
#           .filterDate('2015-01-01', S1_FILTER_END)
#           .filter(ee.Filter.eq('instrumentMode','IW'))
#           .filter(ee.Filter.eq('orbitProperties_pass','DESCENDING'))
#           .filter(ee.Filter.listContains('transmitterReceiverPolarisation','VV')))
# print(f'DESCENDANTE : {desc.size().getInfo()} images')

In [ ]:
# ── CELL 8 — Validation Backscatter VV aux Points Terrain ───────────────────
# TEST DÉCISIF : est-ce que le SAR sépare clairement urbain et sable ?

def build_s1_comp(d1, d2):
    col = s1_col.filterDate(d1,d2).select(['VV','VH'])
    n   = col.size().getInfo()
    print(f'  {d1} → {d2}: {n} images')
    return col.median(), n

print('Composites SAR :')
s1_before, n_before = build_s1_comp(BASE_START,   BASE_END)      # début (config)
s1_after,  n_after  = build_s1_comp(RECENT_START, RECENT_END)    # récent (config)

records_s1 = []
print(f'\nVV Backscatter (dB) aux points terrain :')
print(f'{"Point":12s} {"Type":6s} {"VV_début":>11s} {"VV_récent":>11s} {"ΔVV":>8s}')
print('-' * 55)
for name, info in GROUND_TRUTH.items():
    pt  = ee.Geometry.Point([info['lon'],info['lat']]).buffer(50)
    vb  = s1_before.select(['VV','VH']).reduceRegion(ee.Reducer.mean(),pt,10,maxPixels=1e6).getInfo()
    va  = s1_after.select(['VV','VH']).reduceRegion(ee.Reducer.mean(),pt,10,maxPixels=1e6).getInfo()
    vv_b = round(float(vb['VV']),2); vv_a = round(float(va['VV']),2)
    records_s1.append({'point':name,'type':info['type'],'VV_before':vv_b,'VV_after':vv_a,
                       'VH_before':round(float(vb['VH']),2),'VH_after':round(float(va['VH']),2)})
    print(f'{name:12s} {info["type"]:6s} {vv_b:>11.2f} {vv_a:>11.2f} {vv_a-vv_b:>8.2f}')

df_s1 = pd.DataFrame(records_s1).set_index('point')
urban_vv = df_s1[df_s1.type=='urban']['VV_after'].mean()
sand_vv  = df_s1[df_s1.type=='sand']['VV_after'].mean()
print(f'\n🎯 VV moyen Urbain : {urban_vv:.2f} dB  |  Sable : {sand_vv:.2f} dB')
print(f'   Écart           : {abs(urban_vv-sand_vv):.1f} dB  (vs ~0.05 en NDBI optique)')
print(f'   Seuil optimal   : -14 dB (milieu du gap de {abs(urban_vv-sand_vv):.0f} dB)')

---
## 🏙️ PARTIE 5 — Classification Urbaine SAR (Seuil -14 dB)

### Règle de classification
```
pixel(x,y) = Urbain      si  σ⁰_VV(x,y) > -14 dB  ET  NDWI < 0 (pas d'eau)
           = Non-urbain   sinon
```

Le masque eau (NDWI < 0 depuis Sentinel-2) exclut la sebkha côtière  
qui génère des faux positifs en raison de sa faible rétrodiffusion.

In [ ]:
# ── CELL 9 — Masque Urbain + Nouvelle Construction ──────────────────────────
URBAN_THRESHOLD_DB = -14.0

# Masque eau depuis Sentinel-2 composite récent
b3_s2 = comp_s2.select('B3'); b8_s2 = comp_s2.select('B8')
ndwi  = b3_s2.subtract(b8_s2).divide(b3_s2.add(b8_s2).add(1e-10))
water_mask = ndwi.lt(0.0)   # 1=terre, 0=eau/sebkha

# Masque urbain courant (2021-2024)
urban_mask = s1_after.select('VV').gt(URBAN_THRESHOLD_DB).updateMask(water_mask).rename('urban')

# Nouvelle construction : non-urbain 2016-2018 → urbain 2021-2024
new_construction = (s1_before.select('VV').lt(URBAN_THRESHOLD_DB)
                    .And(s1_after.select('VV').gt(URBAN_THRESHOLD_DB))
                    .rename('new_construction'))

# Validation aux points
print(f'Validation seuil {URBAN_THRESHOLD_DB} dB + masque eau NDWI :')
print(f'{"Point":12s} {"Type":6s} {"VV_après":>10s} {"Classé":>12s}  Correct?')
print('-' * 52)
for _, row in df_s1.iterrows():
    vv  = row['VV_after']
    cls = 'urbain' if vv > URBAN_THRESHOLD_DB else 'non-urbain'
    exp = row['type'] in ['urban','road']
    print(f'{row.name:12s} {row["type"]:6s} {vv:>10.2f} {cls:>12s}  {"✓" if (cls=="urbain")==exp else "✗"}')

# Stats zonales
print('\nFraction urbaine par moughataa :')
print(f'{"Moughataa":20s} {"Urbain%":>9s} {"NvConstr%":>11s} {"VV_moy(dB)":>12s}')
print('-' * 58)
results_sar = []
for _, row in gdf.iterrows():
    geom_ee = ee.Geometry(_json.loads(gpd.GeoSeries([row.geometry],crs='EPSG:4326').to_json())['features'][0]['geometry'])
    uf = urban_mask.reduceRegion(ee.Reducer.mean(),geom_ee,10,maxPixels=1e10,bestEffort=True).getInfo().get('urban',0) or 0
    nc = new_construction.reduceRegion(ee.Reducer.mean(),geom_ee,10,maxPixels=1e10,bestEffort=True).getInfo().get('new_construction',0) or 0
    vv = s1_after.select('VV').reduceRegion(ee.Reducer.mean(),geom_ee,10,maxPixels=1e10,bestEffort=True).getInfo().get('VV',float('nan'))
    results_sar.append({'moughataa':row['ADM2_EN'],'urban_frac':round(uf*100,1),'new_constr_frac':round(nc*100,1),'vv_mean_db':round(float(vv),2) if vv else float('nan')})
    print(f"{row['ADM2_EN']:20s} {uf*100:>8.1f}% {nc*100:>10.1f}% {float(vv):>12.2f}")
df_sar = pd.DataFrame(results_sar)

---
## 🗺️ PARTIE 6 — Visualisations SAR Multi-Années (2016→2025)

Cette section produit **4 cartes distinctes** pour documenter visuellement l'expansion urbaine :

1. `gt_04_sar_urban_map.png` — Vue principale 4 panneaux (RGB · VV · Masque · Nouvelle construction)  
2. `gt_04b_sar_evolution_annuelle.png` — VV brut par année + masques 2016/2020/2024
3. `gt_04c_sar_2016_vs_2024.png` — Comparaison directe + ΔBackscatter
4. `gt_04d_sar_zoom_frontier.png` — Zoom sur les zones de front d'urbanisation (Dar Naim, Toujounine, Ksar)

In [ ]:
# ── CELL 10 — Fonctions Utilitaires + Téléchargement Thumbnails ─────────────
minx, miny, maxx, maxy = bounds

def get_thumb(image, region, bands, vmin, vmax, dims=700, gamma=1.0):
    url = image.getThumbURL({'bands':bands,'min':vmin,'max':vmax,'gamma':gamma,
                              'region':region,'dimensions':dims,'format':'png'})
    return Image.open(BytesIO(requests.get(url, timeout=90).content))

def get_colored_thumb(image, band, region, palette, vmin, vmax, dims=700):
    url = image.select(band).getThumbURL({'min':vmin,'max':vmax,'palette':palette,
                                           'region':region,'dimensions':dims,'format':'png'})
    return Image.open(BytesIO(requests.get(url, timeout=90).content))

def annotate_map(ax, gdf, bounds, buf=0.02, lc='white'):
    ax.set_xlim(bounds[0]-buf, bounds[2]+buf); ax.set_ylim(bounds[1]-buf, bounds[3]+buf)
    gdf.boundary.plot(ax=ax, color='yellow', linewidth=1.2, alpha=0.9)
    for _, row in gdf.iterrows():
        ax.annotate(row['ADM2_EN'], (row.geometry.centroid.x, row.geometry.centroid.y),
                    fontsize=7, color=lc, ha='center', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2',facecolor='black',alpha=0.5,linewidth=0))
    ax.set_xticks([]); ax.set_yticks([])

def plot_gt(ax):
    for name, info in GROUND_TRUTH.items():
        ax.plot(info['lon'], info['lat'], 'o', color=TYPE_COLORS[info['type']],
                ms=8, markeredgecolor='white', markeredgewidth=1.2)

PALETTE_VV    = ['000000','404040','808080','c0c0c0','ffffff']
PALETTE_URBAN = ['1a1a2e','e63946']
PALETTE_DIFF  = ['1d3557','457b9d','f0f4f8','f4a261','e63946']

print('✓ Fonctions définies')

In [ ]:
# ── CELL 11 — Carte SAR Principale (4 panneaux) ─────────────────────────────
print('Téléchargement thumbnails principaux...')
rgb_s2    = get_thumb(comp_s2, region_ee, ['B4','B3','B2'], 0.02, 0.30, gamma=1.4)
vv_grey   = get_colored_thumb(s1_after,'VV',region_ee, PALETTE_VV, -25, -5)
urban_red = get_colored_thumb(urban_mask,'urban',region_ee, ['f0f0f0','e63946'], 0, 1)
new_green = get_colored_thumb(new_construction,'new_construction',region_ee, ['f0f0f0','2a9d8f'], 0, 1)
print('✓ OK')

fig, axes = plt.subplots(1, 4, figsize=(24, 9))
fig.suptitle('Cartographie SAR de l\'Urbanisation — Nouakchott\n'
             'Sentinel-1 VV  |  Seuil -14 dB  |  Composite 2021-2024',
             fontsize=13, fontweight='bold')

for ax, (img, title) in zip(axes, [
    (rgb_s2,    'Sentinel-2 RGB (2023/24)\nRéférence visuelle'),
    (vv_grey,   'Backscatter VV Sentinel-1\nBlanc=fort=urbain | Noir=faible=sable'),
    (urban_red, 'Masque Urbain (VV > -14 dB)\nRouge = zone bâtie'),
    (new_green, 'Nouvelle Construction 2016→2024\nVert = pixels nouvellement bâtis'),
]):
    ax.imshow(img, extent=[minx,maxx,miny,maxy], origin='upper')
    ax.set_aspect('equal'); annotate_map(ax,gdf,bounds); plot_gt(ax)
    ax.set_title(title, fontsize=9, fontweight='bold')

legend_els = [Line2D([0],[0],marker='o',color='w',markerfacecolor=TYPE_COLORS[t],markersize=8,label=t.capitalize())
              for t in ['urban','sand','water','road']]
axes[0].legend(handles=legend_els, loc='lower left', fontsize=8, framealpha=0.8)
plt.tight_layout()
plt.savefig('../outputs/gt_04_sar_urban_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: gt_04_sar_urban_map.png')

In [ ]:
# ── CELL 12 — Évolution Annuelle SAR (années jalons) ────────────────────────
print('Construction composites SAR annuels...')
s1_years = {}
for yr in MILESTONE_YEARS:
    img = s1_col.filterDate(f'{yr}-01-01',f'{yr}-12-31').select('VV').median()
    s1_years[yr] = img

print('Téléchargement VV et masques urbains par année...')
vv_imgs    = {yr: get_colored_thumb(img,'VV',region_ee,PALETTE_VV,-25,-5,dims=600)
              for yr, img in s1_years.items()}
urban_imgs = {}
for yr, img in s1_years.items():
    u_img = img.gt(URBAN_THRESHOLD_DB).updateMask(water_mask).rename('urban')
    urban_imgs[yr] = get_colored_thumb(u_img,'urban',region_ee,PALETTE_URBAN,0,1,dims=600)
print('✓ OK')

fig, axes = plt.subplots(2, 5, figsize=(28, 11))
fig.suptitle(f'Évolution du Backscatter SAR VV — Nouakchott {MILESTONE_YEARS[0]}-{MILESTONE_YEARS[-1]}\n'
             'Ligne 1 : VV brut (plus blanc = plus bâti)  ·  '
             'Ligne 2 : Masque urbain VV > -14 dB (rouge = bâti)',
             fontsize=13, fontweight='bold')

for i, yr in enumerate(MILESTONE_YEARS):
    ax = axes[0,i]
    ax.imshow(vv_imgs[yr], extent=[minx,maxx,miny,maxy], origin='upper')
    ax.set_aspect('equal'); annotate_map(ax,gdf,bounds,lc='yellow')
    ax.set_title(f'VV Backscatter {yr}', fontsize=10, fontweight='bold')

# Ligne 2 : masques début / milieu / fin (colonnes 0, 2, 4)
for pos, yr in [(0, MILESTONE_YEARS[0]), (2, MILESTONE_YEARS[2]), (4, MILESTONE_YEARS[-1])]:
    ax = axes[1,pos]
    ax.imshow(urban_imgs[yr], extent=[minx,maxx,miny,maxy], origin='upper')
    ax.set_aspect('equal'); annotate_map(ax,gdf,bounds,lc='white')
    ax.set_title(f'Masque Urbain {yr}', fontsize=10, fontweight='bold')

for pos in [1,3]:
    axes[1,pos].axis('off')
    axes[1,pos].annotate('', xy=(0.85,0.5), xytext=(0.15,0.5),
                         xycoords='axes fraction',
                         arrowprops=dict(arrowstyle='->',color='#e63946',lw=5,mutation_scale=35))
    axes[1,pos].text(0.5, 0.3, 'Expansion\nurbaine', ha='center', va='center',
                     transform=axes[1,pos].transAxes,
                     fontsize=12, color='#e63946', fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/gt_04b_sar_evolution_annuelle.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: gt_04b_sar_evolution_annuelle.png')

In [ ]:
# ── CELL 13 — Comparaison Directe début vs fin de série + ΔBackscatter ──────
y0, y1 = MILESTONE_YEARS[0], MILESTONE_YEARS[-1]
print(f'Calcul de la différence de backscatter {y0} vs {y1}...')
s1_start = s1_years[y0]; s1_end = s1_years[y1]
vv_diff  = s1_end.subtract(s1_start).rename('vv_diff')

vv_start_img = vv_imgs[y0]
vv_end_img   = vv_imgs[y1]
um_start_img = urban_imgs[y0]
vv_diff_img  = get_colored_thumb(vv_diff,'vv_diff',region_ee,PALETTE_DIFF,-5,5)
print('✓ OK')

fig, axes = plt.subplots(1, 4, figsize=(24, 9))
fig.suptitle(f'Comparaison Directe — Empreinte Urbaine SAR {y0} vs {y1}\n'
             'Expansion de la superficie bâtie détectée par Sentinel-1 VV',
             fontsize=13, fontweight='bold')

for ax, (img, title) in zip(axes, [
    (vv_start_img, f'VV Backscatter {y0}\n(début de série)'),
    (vv_end_img,   f'VV Backscatter {y1}\n(état actuel)'),
    (um_start_img, f'Masque Urbain {y0}\nVV > -14 dB'),
    (vv_diff_img,  f'ΔBackscatter {y0}→{y1}\nRouge=hausse | Bleu=baisse'),
]):
    ax.imshow(img, extent=[minx,maxx,miny,maxy], origin='upper')
    ax.set_aspect('equal'); annotate_map(ax,gdf,bounds); plot_gt(ax)
    ax.set_title(title, fontsize=9, fontweight='bold')

axes[3].text(0.02, 0.04, 'Rouge = nouvelles constructions\nBleu = perte de signal',
             transform=axes[3].transAxes, fontsize=8.5, color='white',
             bbox=dict(facecolor='black',alpha=0.65,boxstyle='round'))

plt.tight_layout()
plt.savefig(f'../outputs/gt_04c_sar_{y0}_vs_{y1}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved: gt_04c_sar_{y0}_vs_{y1}.png')

In [ ]:
# ── CELL 14 — Zoom Fronts d'Urbanisation (Dar Naim · Toujounine · Ksar) ─────
# Zoom sur les 3 moughataat à plus forte croissance — avancée du front bâti
y0, y1 = MILESTONE_YEARS[0], MILESTONE_YEARS[-1]      # 2016, 2025
s1_start = s1_years[y0]; s1_end = s1_years[y1]        # (indépendant de CELL 13)

target_zones   = ['Dar Naim', 'Toujounine', 'Ksar']
# deltas indicatifs (réf. 2016→2024) — actualiser depuis df_ts après CELL 15 si besoin
zone_subtitles = {'Dar Naim':'+6.0pp', 'Toujounine':'+8.8pp', 'Ksar':'+16.8pp'}

fig, axes = plt.subplots(3, 3, figsize=(22, 22))
fig.suptitle(f'Zoom sur les Fronts d\'Urbanisation — Dar Naim · Toujounine · Ksar\n'
             f'SAR VV Backscatter & Masque Urbain {y0} vs {y1}',
             fontsize=13, fontweight='bold')

for row_i, zone in enumerate(target_zones):
    zone_geom = gdf[gdf['ADM2_EN']==zone].geometry.values[0]
    z_minx, z_miny, z_maxx, z_maxy = zone_geom.bounds
    zbuf  = 0.01
    z_reg = ee.Geometry.Rectangle([z_minx-zbuf, z_miny-zbuf, z_maxx+zbuf, z_maxy+zbuf])
    z_ext = [z_minx-zbuf, z_maxx+zbuf, z_miny-zbuf, z_maxy+zbuf]

    vv_start_z = get_colored_thumb(s1_start,'VV',z_reg,PALETTE_VV,-25,-5,dims=500)
    vv_end_z   = get_colored_thumb(s1_end,  'VV',z_reg,PALETTE_VV,-25,-5,dims=500)
    um_end_z   = get_colored_thumb(
        s1_end.gt(URBAN_THRESHOLD_DB).updateMask(water_mask).rename('urban'),
        'urban',z_reg,PALETTE_URBAN,0,1,dims=500)

    for col_i, (img, subtitle) in enumerate([
        (vv_start_z, f'{zone} — VV {y0}'),
        (vv_end_z,   f'{zone} — VV {y1}'),
        (um_end_z,   f'{zone} — Masque Urbain {y1}  ({zone_subtitles[zone]} en {y1-y0} ans)'),
    ]):
        ax = axes[row_i, col_i]
        ax.imshow(img, extent=[z_minx-zbuf,z_maxx+zbuf,z_miny-zbuf,z_maxy+zbuf], origin='upper')
        ax.set_aspect('equal')
        gpd.GeoDataFrame(geometry=[zone_geom]).boundary.plot(ax=ax, color='yellow', lw=1.5)
        ax.set_xlim(z_minx-zbuf, z_maxx+zbuf); ax.set_ylim(z_miny-zbuf, z_maxy+zbuf)
        ax.set_title(subtitle, fontsize=9, fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig('../outputs/gt_04d_sar_zoom_fronts.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: gt_04d_sar_zoom_fronts.png')

# Bar charts résumé (df_sar = composite récent, cf. CELL 9)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Métriques SAR par Moughataa — Nouakchott {RECENT_START[:4]}-{RECENT_END[:4]}',
             fontsize=12, fontweight='bold')
df_plot  = df_sar.sort_values('urban_frac', ascending=True)
df_plot2 = df_sar.sort_values('new_constr_frac', ascending=True)

ax = axes[0]
colors_b = ['#e63946' if v>60 else '#f4a261' if v>35 else '#adb5bd' for v in df_plot['urban_frac']]
ax.barh(df_plot['moughataa'], df_plot['urban_frac'], color=colors_b)
ax.axvline(50,color='black',lw=1,ls='--',alpha=0.5)
ax.set_xlabel('Fraction urbaine (%)'); ax.set_title('Fraction Urbaine (VV > -14 dB)', fontweight='bold')
ax.grid(axis='x',alpha=0.3,ls=':')
for i,v in enumerate(df_plot['urban_frac']): ax.text(v+0.5,i,f'{v:.1f}%',va='center',fontsize=8)

ax = axes[1]
colors_nc = ['#2a9d8f' if v>8 else '#80cbc4' if v>5 else '#adb5bd' for v in df_plot2['new_constr_frac']]
ax.barh(df_plot2['moughataa'], df_plot2['new_constr_frac'], color=colors_nc)
ax.set_xlabel('Nouvelle construction (%)')
ax.set_title(f'Nouvelle Construction {BASE_START[:4]}→{RECENT_END[:4]}', fontweight='bold')
ax.grid(axis='x',alpha=0.3,ls=':')
for i,v in enumerate(df_plot2['new_constr_frac']): ax.text(v+0.1,i,f'{v:.1f}%',va='center',fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/gt_05_sar_bar_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: gt_05_sar_bar_charts.png')

---
## 📈 PARTIE 7 — Série Temporelle 2016-2024 par Moughataa

Pour chaque année et chaque moughataa :
```
f_urbain(zone, t) = Σ(pixels VV > -14 dB dans zone) / Σ(pixels totaux dans zone) × 100
```

La fraction urbaine annuelle est l'input principal de l'indice composite.

In [ ]:
# ── CELL 15 — Extraction Fraction Urbaine Annuelle ──────────────────────────
# YEARS défini en CELL 1b → [2016 .. ANNUAL_END(2025)]. Pas de 2026 (année incomplète).
print(f'Extraction annuelle | Seuil {URBAN_THRESHOLD_DB} dB | Masque eau | Scale 10m')
print(f'{"":22s}', end='')
for y in YEARS: print(f'{y:>8d}', end='')
print(); print('-'*(22+8*len(YEARS)))

records_ts = []
for _, row in gdf.iterrows():
    zone    = row['ADM2_EN']
    geom_ee = ee.Geometry(_json.loads(gpd.GeoSeries([row.geometry],crs='EPSG:4326').to_json())['features'][0]['geometry'])
    row_ts  = {'moughataa': zone}
    print(f'{zone:22s}', end='', flush=True)
    for year in YEARS:
        comp_y = s1_col.filterDate(f'{year}-01-01',f'{year}-12-31').select('VV').median()
        u_y    = comp_y.gt(URBAN_THRESHOLD_DB).updateMask(water_mask).rename('urban')
        frac   = u_y.reduceRegion(ee.Reducer.mean(),geom_ee,10,maxPixels=1e10,bestEffort=True).getInfo().get('urban',None)
        val    = round(frac*100,1) if frac is not None else float('nan')
        row_ts[year] = val; print(f'{val:>8.1f}', end='', flush=True)
    records_ts.append(row_ts); print()

df_ts = pd.DataFrame(records_ts).set_index('moughataa')
print('\nSérie temporelle :')
print(df_ts.to_string())
df_ts.to_csv('../outputs/sar_urban_fraction_timeseries.csv')
print('\n✓ Saved: sar_urban_fraction_timeseries.csv')

In [ ]:
# ── CELL 16 — Visualisation Série Temporelle ────────────────────────────────
years = df_ts.columns.tolist()

# Figure 1 : 9 panneaux individuels
fig, axes = plt.subplots(3, 3, figsize=(18, 14), sharex=True)
fig.suptitle('Fraction Urbaine SAR — Nouakchott par Moughataa 2016-2024\n'
             'Sentinel-1 VV > -14 dB · Composite annuel médian · 10m',
             fontsize=14, fontweight='bold')
summary = []
for ax, zone in zip(axes.flat, ZONE_ORDER):
    vals  = df_ts.loc[zone].values.astype(float)
    color = ZONE_COLORS[zone]
    ax.fill_between(years, vals, alpha=0.12, color=color)
    ax.plot(years, vals, color=color, lw=2, marker='o', ms=5)
    z_fit = np.polyfit(years, vals, 1)
    ax.plot(years, np.poly1d(z_fit)(years), color=color, lw=1.5, ls='--', alpha=0.7)
    total = vals[-1]-vals[0]; rate = total/(len(years)-1)
    ax.annotate(f'Δ=+{total:.1f}pp\n{rate:+.2f}pp/an',
                xy=(years[-1],vals[-1]), xytext=(-65,-28), textcoords='offset points',
                fontsize=8.5, color=color, fontweight='bold',
                arrowprops=dict(arrowstyle='->',color=color,lw=1.0))
    ax.set_title(zone, fontsize=11, fontweight='bold', color=color)
    ax.set_ylim(max(0,vals.min()-8), min(105,vals.max()+8))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'{v:.0f}%'))
    ax.set_xticks(years); ax.set_xticklabels([str(y) for y in years], rotation=45, fontsize=7.5)
    ax.grid(alpha=0.25, ls=':')
    summary.append({'zone':zone,'v2016':round(float(vals[0]),1),'v2024':round(float(vals[-1]),1),
                    'change':round(total,1),'rate':round(rate,2)})
plt.tight_layout()
plt.savefig('../outputs/gt_06_timeseries_panels.png', dpi=150, bbox_inches='tight')
plt.show()

# Figure 2 : toutes zones sur un graphique
fig, ax = plt.subplots(figsize=(14, 7))
fig.suptitle('Fraction Urbaine — Toutes Moughataat · Sentinel-1 SAR 2016-2024', fontsize=13, fontweight='bold')
for zone in ZONE_ORDER:
    vals = df_ts.loc[zone].values.astype(float); color = ZONE_COLORS[zone]
    ax.plot(years, vals, color=color, lw=2.2, marker='o', ms=5)
    ax.annotate(zone, xy=(years[-1],vals[-1]), xytext=(4,0), textcoords='offset points',
                fontsize=8.5, color=color, fontweight='bold', va='center')
city_avg = df_ts.mean(axis=0).values
ax.plot(years, city_avg, color='black', lw=2.5, ls='-.', marker='D', ms=5)
ax.annotate('Moy. ville', xy=(years[-1],city_avg[-1]), xytext=(4,0), textcoords='offset points',
            fontsize=8.5, color='black', fontweight='bold', va='center')
ax.set_ylabel('Fraction Urbaine (%)'); ax.set_xlabel('Année')
ax.set_xticks(years); ax.set_xticklabels([str(y) for y in years], fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'{v:.0f}%'))
ax.grid(alpha=0.25, ls=':'); ax.set_xlim(years[0]-0.3, years[-1]+2.5)
plt.tight_layout()
plt.savefig('../outputs/gt_07_timeseries_combined.png', dpi=150, bbox_inches='tight')
plt.show()

df_summary = pd.DataFrame(summary).set_index('zone')
print('Résumé Croissance 2016→2024')
print('='*72)
print(f'{"Moughataa":20s} {"2016%":>7s} {"2024%":>7s} {"Δpp":>7s} {"pp/an":>7s}  Trajectoire')
print('-'*72)
for _, r in df_summary.iterrows():
    traj = 'Rapide' if r['rate']>=1.5 else 'Modérée' if r['rate']>=0.8 else 'Lente'
    print(f"{r.name:20s} {r['v2016']:>7.1f} {r['v2024']:>7.1f} {r['change']:>+7.1f} {r['rate']:>+7.2f}  {traj}")
print('='*72)
print(f'{"Ville":20s} {df_ts[2016].mean():>7.1f} {df_ts[2024].mean():>7.1f} {df_ts[2024].mean()-df_ts[2016].mean():>+7.1f} {(df_ts[2024].mean()-df_ts[2016].mean())/8:>+7.2f}')

---
## 📊 PARTIE 8 — Indice Composite & Indice Synthétique Ville

### Pondération
| Composante | Mesure | Poids |
|-----------|--------|:-----:|
| C1 — Fraction urbaine 2024 | % de la zone bâtie | **40%** |
| C2 — Taux de croissance | pp/an sur 8 ans | **30%** |
| C3 — Intensité construction | nouveaux pixels / total | **20%** |
| C4 — Densité backscatter SAR | VV moyen 2021-2024 (dB) | **10%** |

Toutes les composantes sont normalisées **Min-Max [0, 100]** avant pondération,  
ce qui rend l'indice **relatif** entre les 9 moughataat.

In [ ]:
# ── CELL 17 — Calcul Indice Composite ───────────────────────────────────────
constr_intensity = dict(zip(df_sar['moughataa'], df_sar['new_constr_frac']))
vv_means         = dict(zip(df_sar['moughataa'], df_sar['vv_mean_db']))

def normalize_series(d):
    vals = np.array(list(d.values()), dtype=float)
    mn, mx = vals.min(), vals.max()
    return {k: round(100*(v-mn)/(mx-mn+1e-10),2) for k,v in d.items()}

# NB : variable gardée sous le nom 'uf_2024' pour ne pas casser CELL 18/19/11H
#      qui la référencent — elle contient désormais l'UF de ANNUAL_END (2025).
uf_2024  = {z: df_ts.loc[z, ANNUAL_END] for z in ZONE_ORDER}
gr_total = {z: df_summary.loc[z,'rate'] for z in ZONE_ORDER}

uf_norm = normalize_series(uf_2024)
gr_norm = normalize_series(gr_total)
ci_norm = normalize_series(constr_intensity)
vv_norm = normalize_series(vv_means)

W_UF, W_GR, W_CI, W_VV = 0.40, 0.30, 0.20, 0.10
composite = {z: round(W_UF*uf_norm[z]+W_GR*gr_norm[z]+W_CI*ci_norm[z]+W_VV*vv_norm[z],2) for z in ZONE_ORDER}

print('Indice Composite — Décomposition')
print('='*80)
print(f'{"Zone":20s} {"UF(40%)":>9s} {"GR(30%)":>9s} {"CI(20%)":>9s} {"VV(10%)":>9s} {"INDICE":>8s}')
print('-'*80)
for zone in sorted(composite, key=composite.get, reverse=True):
    print(f'{zone:20s} {uf_norm[zone]:>9.1f} {gr_norm[zone]:>9.1f} {ci_norm[zone]:>9.1f} {vv_norm[zone]:>9.1f} {composite[zone]:>8.1f}')
print('='*80)

gdf_proj   = gdf.to_crs(epsg=32628)
areas      = {row['ADM2_EN']: row.geometry.area/1e6 for _, row in gdf_proj.iterrows()}
total_area = sum(areas.values())
city_index_snap = sum(composite[z]*areas[z]/total_area for z in ZONE_ORDER)
print(f'\nIndice Synthétique Ville (pondéré superficie) : {city_index_snap:.1f} / 100')
print()
for z,a in sorted(areas.items(), key=lambda x: x[1], reverse=True):
    print(f'  {z:20s}: {a:6.2f} km² ({a/total_area*100:.1f}%)')

In [ ]:
# ── CELL 18 — Visualisations de l'Indice ────────────────────────────────────
cmap_idx = LinearSegmentedColormap.from_list('urban_idx',['#d62728','#f4a261','#ffffb2','#74c476','#238b45'])

# Figure 1 : Cartes choroplèthes
fig, axes = plt.subplots(1, 3, figsize=(20, 9))
fig.suptitle('Indice Urbain Composite — Nouakchott par Moughataa\n'
             'Fraction(40%) + Croissance(30%) + Construction(20%) + Densité SAR(10%)',
             fontsize=13, fontweight='bold')

gdf_idx = gdf.copy()
gdf_idx['index']    = gdf_idx['ADM2_EN'].map(composite)
gdf_idx['uf_2024']  = gdf_idx['ADM2_EN'].map(uf_2024)
gdf_idx['gr_total'] = gdf_idx['ADM2_EN'].map({z:df_summary.loc[z,'rate'] for z in ZONE_ORDER})

for ax, (col, title, cmap, vmin, vmax) in zip(axes, [
    ('uf_2024', 'Fraction Urbaine 2024 (%)',  'YlOrRd', 30, 100),
    ('gr_total','Taux de Croissance (pp/an)', 'YlGn',    0, 2.5),
    ('index',   'Indice Composite (0-100)',   cmap_idx,  0, 100),
]):
    gdf_idx.plot(column=col, ax=ax, cmap=cmap, vmin=vmin, vmax=vmax,
                 edgecolor='white', linewidth=1.5)
    for _, row in gdf_idx.iterrows():
        v = row[col]
        fmt  = f'{v:.1f}' if col=='gr_total' else f'{v:.0f}'
        unit = 'pp/an' if col=='gr_total' else ('%' if col=='uf_2024' else '')
        ax.annotate(f"{row['ADM2_EN']}\n{fmt}{unit}",
                    (row.geometry.centroid.x, row.geometry.centroid.y),
                    fontsize=8, ha='center', va='center', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.25',facecolor='white',alpha=0.75,linewidth=0))
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin,vmax=vmax))
    sm.set_array([]); plt.colorbar(sm, ax=ax, shrink=0.6, pad=0.02)
    ax.set_title(title, fontsize=11, fontweight='bold'); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig('../outputs/idx_01_choropleth.png', dpi=150, bbox_inches='tight')
plt.show()

# Figure 2 : Barres empilées + évolution annuelle approchée
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle("Décomposition de l'Indice Urbain Composite par Moughataa", fontsize=13, fontweight='bold')

zones_ranked = sorted(composite, key=composite.get, reverse=True)
yr = range(len(zones_ranked))
uf_c = [W_UF*uf_norm[z] for z in zones_ranked]
gr_c = [W_GR*gr_norm[z] for z in zones_ranked]
ci_c = [W_CI*ci_norm[z] for z in zones_ranked]
vv_c = [W_VV*vv_norm[z] for z in zones_ranked]
ax = axes[0]
ax.barh(yr, uf_c, color='#e63946', label='Fraction urbaine (40%)', alpha=0.9)
ax.barh(yr, gr_c, left=uf_c, color='#2a9d8f', label='Croissance (30%)', alpha=0.9)
l2 = [a+b for a,b in zip(uf_c,gr_c)]
ax.barh(yr, ci_c, left=l2, color='#f4a261', label='Construction (20%)', alpha=0.9)
l3 = [a+b for a,b in zip(l2,ci_c)]
ax.barh(yr, vv_c, left=l3, color='#457b9d', label='Densité SAR (10%)', alpha=0.9)
for i, zone in enumerate(zones_ranked):
    ax.text(composite[zone]+0.5, i, f'{composite[zone]:.1f}', va='center', fontsize=9, fontweight='bold')
ax.set_yticks(list(yr)); ax.set_yticklabels(zones_ranked, fontsize=10)
ax.set_xlabel('Score (0-100)'); ax.set_title('Contributions par composante', fontweight='bold')
ax.legend(loc='lower right', fontsize=8); ax.grid(axis='x', alpha=0.3, ls=':'); ax.set_xlim(0,75)

uf_min, uf_max = min(uf_2024.values()), max(uf_2024.values())
ax = axes[1]
for zone in ZONE_ORDER:
    color   = ZONE_COLORS[zone]
    uf_vals = df_ts.loc[zone].values.astype(float)
    uf_n_y  = [(v-uf_min)/(uf_max-uf_min+1e-10)*100 for v in uf_vals]
    static  = W_GR*gr_norm[zone]+W_CI*ci_norm[zone]+W_VV*vv_norm[zone]
    ann_idx = [W_UF*u+static for u in uf_n_y]
    ax.plot(years, ann_idx, color=color, lw=2, marker='o', ms=4)
    ax.annotate(zone, xy=(years[-1],ann_idx[-1]), xytext=(3,0), textcoords='offset points',
                fontsize=8, color=color, fontweight='bold', va='center')
ax.set_xlabel('Année'); ax.set_ylabel('Indice composite (approx.)')
ax.set_title('Évolution annuelle\n(composante UF variable, autres fixes)', fontweight='bold', fontsize=10)
ax.set_xticks(years); ax.set_xticklabels([str(y) for y in years], rotation=45, fontsize=8)
ax.grid(alpha=0.3, ls=':'); ax.set_xlim(years[0]-0.3, years[-1]+2.5)
plt.tight_layout()
plt.savefig('../outputs/idx_02_index_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: idx_01_choropleth.png, idx_02_index_breakdown.png')

In [ ]:
# ── CELL 19 — Indice Synthétique Ville 2016-2024 ────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('Indice Synthétique Ville de Nouakchott — Évolution 2016-2024\n'
             'Moyenne pondérée par superficie des moughataat', fontsize=13, fontweight='bold')

uf_min, uf_max = min(uf_2024.values()), max(uf_2024.values())
city_annual = []
for year in years:
    idx_yr = 0
    for zone in ZONE_ORDER:
        uf_y   = df_ts.loc[zone, year]
        uf_n   = (uf_y-uf_min)/(uf_max-uf_min+1e-10)*100
        static = W_GR*gr_norm[zone]+W_CI*ci_norm[zone]+W_VV*vv_norm[zone]
        idx_yr += (W_UF*uf_n+static)*areas[zone]/total_area
    city_annual.append(round(idx_yr,2))

ax.fill_between(years, city_annual, alpha=0.15, color='#1d3557')
ax.plot(years, city_annual, color='#1d3557', lw=3, marker='D', ms=7, zorder=5)
for y, v in zip(years, city_annual):
    ax.annotate(f'{v:.1f}', xy=(y,v), xytext=(0,10), textcoords='offset points',
                fontsize=9, ha='center', fontweight='bold', color='#1d3557')
z_fit = np.polyfit(years, city_annual, 1)
ax.plot(years, np.poly1d(z_fit)(years), color='#e63946', lw=1.5, ls='--',
        label=f'Tendance: +{z_fit[0]:.2f} pts/an')
total_chg = city_annual[-1]-city_annual[0]
ax.set_ylabel('Indice synthétique (0-100)'); ax.set_xlabel('Année')
ax.set_xticks(years); ax.set_xticklabels([str(y) for y in years], fontsize=10)
ax.grid(alpha=0.3, ls=':'); ax.legend(fontsize=10)
ax.annotate(f'Variation totale: +{total_chg:.1f} pts\nsur 8 ans (2016→2024)',
            xy=(2022,city_annual[6]), xytext=(2019.5,city_annual[0]*0.93),
            fontsize=10, color='#e63946', fontweight='bold',
            arrowprops=dict(arrowstyle='->',color='#e63946',lw=1.5))
plt.tight_layout()
plt.savefig('../outputs/idx_03_city_synthetic_index.png', dpi=150, bbox_inches='tight')
plt.show()

print('Indice synthétique ville :')
for y, v in zip(years, city_annual):
    print(f'  {y}: {v:5.1f}  {"█"*int(v/2)}')
print(f'\nVariation totale : +{total_chg:.1f} pts  |  Taux annuel : +{total_chg/8:.2f} pts/an')

---
## 💹 PARTIE 9 — Données Économiques & Analyse de Corrélation

### Variables
| Variable | Source | Fréquence |
|----------|--------|-----------|
| ICC Global (Base 100=2012) | ANSADE | Trimestriel → annualisé |
| ICC Matériaux | ANSADE | Trimestriel → annualisé |
| ICC Main d'œuvre | ANSADE | Trimestriel → annualisé |
| PIB croissance % | World Bank WDI | Annuel |
| Croissance pop. urbaine % | World Bank WDI | Annuel |

> ⚠️ Avertissement : la corrélation en niveaux SAR/ICC sera ~0.97 (très élevée),  
> mais les deux séries sont non-stationnaires → corrélation potentiellement fallacieuse.

In [ ]:
# ── CELL 20 — Chargement Données Économiques (ICC corrigé + Population) ─────
from scipy import stats

# ═══════════════════════════════════════════════════════════════════════════
# 1. ICC TRIMESTRIEL — source ANSADE, base 100 = 2012. SOURCE DE VÉRITÉ.
#    ⚠️ Remplace l'ancienne série, décalée de 2 trimestres à partir de 2T2020.
# ═══════════════════════════════════════════════════════════════════════════
ICC_LABELS = [f'{q}T{y}' for y in range(2012, 2027) for q in (1,2,3,4)]
ICC_LABELS = ICC_LABELS[ICC_LABELS.index('3T2012'):ICC_LABELS.index('1T2026')+1]

ICC_RAW = {
 'icc_global'    : [100.7,101.1,101.7,102.7,103.6,104.6,105.4,106.1,106.5,107.3,107.7,108.1,108.9,
                    109.2,108.0,107.2,107.8,107.3,104.2,107.3,107.6,107.8,103.5,107.9,110.6,109.0,
                    110.0,113.3,111.7,111.1,108.2,110.4,109.5,109.5,121.1,119.7,125.9,124.8,128.2,
                    136.2,132.9,135.3,135.6,137.9,138.8,138.6,141.7,139.6,142.5,140.0,139.6,145.6,
                    148.1,148.0,149.4],
 'icc_materiaux' : [100.5,100.7,101.1,102.1,103.1,104.1,104.5,105.4,106.2,106.2,106.6,107.0,107.4,
                    107.7,104.1,102.8,102.9,102.5,96.3,100.8,103.0,103.2,97.8,103.4,103.7,102.4,
                    104.6,107.1,104.3,103.5,101.3,103.3,103.2,102.4,114.3,113.9,117.2,116.2,118.7,
                    124.5,120.2,123.2,122.4,124.1,124.4,124.9,126.9,125.5,126.7,124.3,124.5,128.6,
                    130.8,130.5,132.0],
 'icc_gestion'   : [102.9,104.8,105.9,106.5,106.5,106.8,106.8,106.9,107.0,107.0,107.0,107.0,107.0,
                    107.0,108.0,108.0,107.9,107.8,108.0,108.0,108.0,108.0,108.0,108.0,108.0,108.0,
                    108.0,108.0,108.0,108.0,108.1,108.1,101.4,108.2,104.3,104.8,105.2,105.8,106.3,
                    106.5,121.5,125.1,125.2,125.2,125.3,125.4,125.6,125.7,126.5,126.7,126.7,127.0,
                    127.2,127.1,126.8],
 'icc_location'  : [99.1,98.6,98.3,98.0,98.0,97.8,97.7,97.0,97.3,96.5,96.4,95.6,94.9,94.9,94.9,94.8,
                    103.1,104.2,102.8,103.7,105.2,105.1,103.4,103.4,104.6,105.4,92.4,104.1,104.7,
                    103.7,103.7,103.7,94.5,94.5,97.5,99.0,99.8,105.2,106.3,116.4,116.3,116.3,116.9,
                    130.2,155.8,159.6,159.3,154.7,158.9,159.9,164.1,174.8,181.0,190.2,203.5],
 'icc_mainoeuvre': [101.7,102.8,104.5,105.4,106.1,106.9,109.6,109.7,109.1,113.1,113.9,114.2,116.4,
                    117.0,125.5,126.5,128.6,127.4,136.9,134.5,127.1,126.9,126.9,127.0,140.1,136.8,
                    134.3,140.2,143.6,143.6,137.2,140.7,137.3,140.8,152.9,146.9,166.0,163.7,171.4,
                    188.3,187.9,187.8,192.9,196.6,197.0,193.5,201.5,196.9,207.1,203.3,200.2,213.6,
                    217.2,216.7,216.9],
}
for k, v in ICC_RAW.items():
    assert len(v) == len(ICC_LABELS), f'{k}: {len(v)} vs {len(ICC_LABELS)}'

# → DataFrame trimestriel indexé par date (QS = début de trimestre, comme df_q)
icc_dates = pd.to_datetime([f'{lbl[-4:]}-{(int(lbl[0])-1)*3+1:02d}-01' for lbl in ICC_LABELS])
df_icc_q = pd.DataFrame(ICC_RAW, index=icc_dates)
df_icc_q.index.name = 'date'
df_icc_q['label'] = [f'{d.year}-Q{(d.month-1)//3+1}' for d in df_icc_q.index]

# ═══════════════════════════════════════════════════════════════════════════
# 2. ICC ANNUEL — DÉRIVÉ du trimestriel (plus jamais codé en dur)
#    Une année n'est retenue que si ses 4 trimestres sont publiés.
# ═══════════════════════════════════════════════════════════════════════════
icc_cols  = list(ICC_RAW.keys())
_grp      = df_icc_q.groupby(df_icc_q.index.year)
df_icc_a  = _grp[icc_cols].mean().round(2)
df_icc_a  = df_icc_a[_grp[icc_cols[0]].count() == 4]          # années complètes seulement
print('ICC annuel (dérivé du trimestriel, années complètes) :')
print(df_icc_a.loc[YEAR_START:ANNUAL_END].to_string())

# ═══════════════════════════════════════════════════════════════════════════
# 3. POPULATION — RGPH + projections ANSADE (wilaya de Nouakchott)
#    ⚠️ Voir note de prudence : NE PAS utiliser comme régresseur de série temp.
# ═══════════════════════════════════════════════════════════════════════════
POP_NKC = {1977:134704, 1988:393325, 2000:558195, 2013:958399,
           2023:1446761, 2024:1471589, 2025:1514455, 2026:1558058, 2027:1602348}

# Les 3 wilayas de Nouakchott (projections 2023+ uniquement) → groupes de moughataat
POP_WILAYA = {
    'Nouakchott-Ouest': {2023:204881, 2024:205528, 2025:211515, 2026:217605, 2027:223791},
    'Nouakchott-Nord' : {2023:614465, 2024:629149, 2025:647476, 2026:666117, 2027:685052},
    'Nouakchott-Sud'  : {2023:627415, 2024:636912, 2025:655464, 2026:674336, 2027:693505},
}
WILAYA_TO_MOUGHATAA = {                       # découpage administratif standard
    'Nouakchott-Ouest': ['Tevragh Zein', 'Ksar', 'Sebkha'],
    'Nouakchott-Nord' : ['Dar Naim', 'Teyarett', 'Toujounine'],
    'Nouakchott-Sud'  : ['Arafat', 'El Mina', 'Riyadh'],
}

# ═══════════════════════════════════════════════════════════════════════════
# 4. World Bank WDI (annuel) — 2026 n'existera pas avant 2027
# ═══════════════════════════════════════════════════════════════════════════
wb_data = {
    'year'           : list(range(2015, ANNUAL_END + 1)),
    'gdp_growth'     : [5.376,1.261,6.271,4.772,3.141,-0.361,0.743,6.796,6.809,6.308, None],  # ← 2025
    'urb_pop_growth' : [5.353,5.389,5.375,4.681,4.625,4.556,4.485,4.490,4.489,4.368, None],  # ← 2025
}
df_wb = pd.DataFrame(wb_data).set_index('year')

# ═══════════════════════════════════════════════════════════════════════════
# 5. DATASET ANNUEL (compatibilité CELL 21-24)
# ═══════════════════════════════════════════════════════════════════════════
eco_years = list(range(YEAR_START, ANNUAL_END + 1))
df_eco = pd.DataFrame({'year': eco_years}).set_index('year')
df_eco['sar_index']      = pd.Series(dict(zip(years, city_annual)))
df_eco['icc_global']     = df_icc_a['icc_global']
df_eco['icc_materials']  = df_icc_a['icc_materiaux']
df_eco['icc_labor']      = df_icc_a['icc_mainoeuvre']
df_eco['icc_location']   = df_icc_a['icc_location']
df_eco['gdp_growth']     = df_wb['gdp_growth'].reindex(eco_years)
df_eco['urb_pop_growth'] = df_wb['urb_pop_growth'].reindex(eco_years)
df_eco['pop_nkc']        = pd.Series(POP_NKC).reindex(eco_years)   # trous = années non-recensées
df_eco['sar_change']     = df_eco['sar_index'].diff()
df_eco['icc_change']     = df_eco['icc_global'].diff()

print('\nDataset maître ANNUEL :')
print(df_eco.round(3).to_string())

sar    = df_eco['sar_index'].values
sar_chg= df_eco['sar_change'].fillna(0).values
icc    = df_eco['icc_global'].values
gdp_gr = df_eco['gdp_growth'].values
urb_gr = df_eco['urb_pop_growth'].values
labor  = df_eco['icc_labor'].values
# ═══════════════════════════════════════════════════════════════════════════
# 6. Dataset TRIMESTRIEL → construit en CELL 12F (après CELL 12A, qui crée df_q).
#    df_icc_q est prêt ici et sera consommé là-bas.
# ═══════════════════════════════════════════════════════════════════════════
print(f'\n✓ ICC trimestriel prêt : {len(df_icc_q)} trimestres '
      f'({df_icc_q["label"].iloc[0]} → {df_icc_q["label"].iloc[-1]})')
print('  → jointure SAR×ICC en CELL 12F (après CELL 12A)')

In [ ]:
# ── CELL 21 — Lag Correlations (annuel) ─────────────────────────────────────
def lag_correlation(x, y, max_lag=3):
    results = []
    x, y = np.array(x,dtype=float), np.array(y,dtype=float)
    for lag in range(-max_lag, max_lag+1):
        xi, yi = (x[:-lag],y[lag:]) if lag>0 else (x[-lag:],y[:lag]) if lag<0 else (x,y)
        mask   = ~(np.isnan(xi)|np.isnan(yi))
        if mask.sum()>=4:
            r,p = stats.pearsonr(xi[mask],yi[mask])
            results.append({'lag':lag,'r':round(r,3),'p':round(p,3),'n':mask.sum()})
        else:
            results.append({'lag':lag,'r':np.nan,'p':np.nan,'n':0})
    return pd.DataFrame(results).set_index('lag')

locat = df_eco['icc_location'].values          # nouvelle composante (explose : 99 → 178)

corr_icc = lag_correlation(icc,    sar)
corr_gdp = lag_correlation(gdp_gr, sar_chg)
corr_lab = lag_correlation(labor,  sar)
corr_urb = lag_correlation(urb_gr, sar_chg)
corr_loc = lag_correlation(locat,  sar)

for name, c in [('ICC Global → SAR', corr_icc), ('PIB → ΔSAR', corr_gdp),
                ('ICC Main d\'œuvre → SAR', corr_lab), ('Pop. Urbaine → ΔSAR', corr_urb),
                ('ICC Location matériels → SAR', corr_loc)]:
    print(f'{"="*55}\n{name}\n{"="*55}'); print(c.to_string()); print()

print('⚠️  n ≤ 10 → ces corrélations sont indicatives. Test de référence : CELL 12G (n≈41).')

In [ ]:
# ── CELL 22 — Visualisation Corrélation + Matrice ───────────────────────────
fig = plt.figure(figsize=(22, 18))
fig.suptitle('Analyse de Corrélation — Indice SAR vs Variables Économiques\n'
             f'Nouakchott {eco_years[0]}-{eco_years[-1]}  ·  ICC corrigé',
             fontsize=14, fontweight='bold')
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.38)

# Panneau 1 : SAR + ICC double axe
ax1 = fig.add_subplot(gs[0,:2]); ax1b = ax1.twinx()
ax1.plot(eco_years, sar, color='#1d3557', lw=2.5, marker='D', ms=6, label='Indice SAR (gauche)')
ax1b.plot(eco_years, icc, color='#e63946', lw=2.5, marker='s', ms=6, ls='--', label='ICC Global (droite)')
ax1.set_ylabel('Indice SAR (0-100)', color='#1d3557', fontsize=10)
ax1b.set_ylabel('ICC (Base 100=2012)', color='#e63946', fontsize=10)
ax1.tick_params(axis='y',labelcolor='#1d3557'); ax1b.tick_params(axis='y',labelcolor='#e63946')
lines = ax1.get_lines()+ax1b.get_lines()
ax1.legend(lines,[l.get_label() for l in lines],loc='upper left',fontsize=9)
ax1.set_title('SAR vs ICC — Corrélation en niveaux (potentiellement fallacieuse)', fontweight='bold')
ax1.set_xticks(eco_years); ax1.set_xticklabels([str(y) for y in eco_years], rotation=45, fontsize=8)
ax1.grid(alpha=0.25, ls=':')

# Panneau 2 : PIB vs ΔSAR
ax2 = fig.add_subplot(gs[0,2]); ax2b = ax2.twinx()
ax2.bar(eco_years, df_eco['gdp_growth'].values, color='#2a9d8f', alpha=0.5)
ax2b.plot(eco_years, sar_chg, color='#1d3557', lw=2, marker='o', ms=5)
ax2.set_title('PIB growth vs ΔSAR', fontweight='bold', fontsize=9)
ax2.set_ylabel('PIB (%)', fontsize=8, color='#2a9d8f'); ax2b.set_ylabel('ΔSAR', fontsize=8, color='#1d3557')
ax2.tick_params(axis='y',labelcolor='#2a9d8f'); ax2b.tick_params(axis='y',labelcolor='#1d3557')
ax2.set_xticks(eco_years); ax2.set_xticklabels([str(y) for y in eco_years], rotation=45, fontsize=7)
ax2.grid(alpha=0.2, ls=':')

# Panneaux lag correlation
for corr_df, title, color, pos in [
    (corr_icc, 'ICC Global → SAR',        '#e63946', gs[1,0]),
    (corr_gdp, 'PIB growth → ΔSAR',       '#2a9d8f', gs[1,1]),
    (corr_lab, "ICC Main d'œuvre → SAR",  '#f4a261', gs[1,2]),
    (corr_loc, 'ICC Location → SAR',      '#8338ec', gs[2,0]),
]:
    ax = fig.add_subplot(pos)
    lgs = corr_df.index.tolist(); rs = corr_df['r'].tolist(); ps = corr_df['p'].tolist()
    bcolors = ['#e63946' if (not pd.isna(r) and p<0.05) else '#f4a261' if (not pd.isna(r) and p<0.10) else '#dddddd'
               for r,p in zip(rs,ps)]
    bars = ax.bar(lgs, rs, color=bcolors, alpha=0.85, width=0.6)
    ax.axhline(0,color='black',lw=0.8); ax.axhline(0.7,color='green',lw=1,ls='--',alpha=0.4)
    ax.axhline(-0.7,color='green',lw=1,ls='--',alpha=0.4)
    for bar,r,p in zip(bars,rs,ps):
        if pd.isna(r): continue
        star = '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.10 else ''
        if star: ax.text(bar.get_x()+bar.get_width()/2, r+(0.04 if r>=0 else -0.07),
                         star, ha='center', fontsize=9, fontweight='bold')
    ax.set_title(title,fontsize=9,fontweight='bold')
    ax.set_xlabel('Décalage (années)',fontsize=7); ax.set_ylabel('r de Pearson',fontsize=8)
    ax.set_ylim(-1.05,1.05); ax.set_xticks(lgs); ax.set_xticklabels([f'{l:+d}' for l in lgs],fontsize=8)
    ax.grid(axis='y',alpha=0.25,ls=':')

# Matrice de corrélation  ← FIX : icc_materials n'existe plus, on lit df_eco
ax7 = fig.add_subplot(gs[2,1:])
vars_matrix = {
    'SAR index'     : sar,
    'ΔSAR'          : sar_chg,
    'ICC global'    : icc,
    'ICC matériaux' : df_eco['icc_materials'].values,
    'ICC main-œuvre': labor,
    'ICC location'  : locat,
    'PIB growth'    : gdp_gr,
    'Urb. pop Δ'    : urb_gr,
}
vnames = list(vars_matrix.keys()); n = len(vnames)
mat_r = np.full((n,n), np.nan); mat_p = np.full((n,n), np.nan)
for i,(n1,v1) in enumerate(vars_matrix.items()):
    for j,(n2,v2) in enumerate(vars_matrix.items()):
        v1 = np.asarray(v1,dtype=float); v2 = np.asarray(v2,dtype=float)
        mask = ~(np.isnan(v1)|np.isnan(v2))
        if mask.sum()>=4:
            mat_r[i,j], mat_p[i,j] = stats.pearsonr(v1[mask],v2[mask])
cmap_div = LinearSegmentedColormap.from_list('corr',['#1d3557','#457b9d','#f0f4f8','#e63946','#9b1c2a'])
im = ax7.imshow(mat_r, cmap=cmap_div, vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im,ax=ax7,shrink=0.8,label='r de Pearson')
ax7.set_xticks(range(n)); ax7.set_yticks(range(n))
ax7.set_xticklabels(vnames,rotation=35,ha='right',fontsize=8.5)
ax7.set_yticklabels(vnames,fontsize=8.5)
ax7.set_title('Matrice de Corrélation (niveaux — à interpréter avec prudence)\n'
              '*** p<0.01  ** p<0.05  * p<0.10',fontweight='bold',fontsize=10)
for i in range(n):
    for j in range(n):
        r=mat_r[i,j]; p=mat_p[i,j]
        if np.isnan(r): continue
        star = '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.10 else ''
        ax7.text(j,i,f'{r:.2f}{star}',ha='center',va='center',
                 fontsize=7.5,color='white' if abs(r)>0.6 else 'black',fontweight='bold')

plt.savefig('../outputs/idx_04_correlation_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: idx_04_correlation_analysis.png')

---
## 🔬 PARTIE 10 — Tests Économétriques de Robustesse

### Problème de la corrélation fallacieuse
Deux séries tendancielles donnent toujours r ≈ 1 même sans lien causal.

### Tests appliqués
| Test | Question | Résultat attendu |
|------|----------|-----------------|
| **ADF (niveaux)** | Séries stationnaires ? | Non → tendances communes |
| **ADF (Δ)** | Stationnaires après différenciation ? | Oui → I(1) |
| **Corrélation sur Δ** | Lien après détrending ? | r ≈ 0 → pas de co-mouvement réel |
| **Cointégration** | Relation de long terme ? | p=0.002 (mais n=9, faible puissance) |
| **Granger** | ICC prédit-il l'avenir du SAR ? | F=0.055, p=0.826 → NON |

### Conclusion
ICC et SAR sont tous les deux **des symptômes du même boom de construction**,  
non des variables causalement liées entre elles.

In [ ]:
# ── CELL 23 — ADF, Différenciation, Cointégration, Granger (ANNUEL) ─────────
# ⚠️ n≈10 → puissance très faible. Conservée pour continuité ; le test de
#    référence du mémoire est désormais CELL 12G (trimestriel, n≈41).
from statsmodels.tsa.stattools import adfuller, grangercausalitytests, coint
import warnings; warnings.filterwarnings('ignore')

d_sar = np.diff(sar); d_icc = np.diff(icc)
d_lab = np.diff(labor); d_gdp = np.diff(gdp_gr); d_urb = np.diff(urb_gr)
yrs_d = eco_years[1:]

series_levels = {'SAR index':sar,'ICC global':icc,'ICC labor':labor,'GDP growth':gdp_gr,'Urb pop Δ':urb_gr}
series_diffs  = {'ΔSAR':d_sar,'ΔICC':d_icc,'ΔICC labor':d_lab,'ΔGDP':d_gdp,'ΔUrb':d_urb}

adf_lvl, adf_dif = {}, {}
print(f'ÉTAPE 1 — ADF sur NIVEAUX (H0: racine unitaire)   [n={len(sar)}]')
print('='*60)
for name, s in series_levels.items():
    clean = np.asarray(s,dtype=float); clean = clean[~np.isnan(clean)]
    stat, p = adfuller(clean, maxlag=1, autolag=None)[:2]
    adf_lvl[name] = p
    print(f'  {name:20s}  ADF={stat:7.3f}  p={p:.4f}  → {"NON-STAT ⚠️" if p>0.05 else "STAT ✓"}')

print('\nÉTAPE 2 — ADF sur DIFFÉRENCES PREMIÈRES')
print('='*60)
for name, s in series_diffs.items():
    clean = np.asarray(s,dtype=float); clean = clean[~np.isnan(clean)]
    stat, p = adfuller(clean, maxlag=1, autolag=None)[:2]
    adf_dif[name] = p
    print(f'  {name:20s}  ADF={stat:7.3f}  p={p:.4f}  → {"STAT ✓" if p<0.05 else "encore non-stat"}')

print('\nÉTAPE 3 — CORRÉLATION SUR DIFFÉRENCES')
print('='*60)
r_d, p_d = np.nan, np.nan
for label, x, y in [('ΔSAR vs ΔICC',d_sar,d_icc),('ΔSAR vs ΔICC labor',d_sar,d_lab),
                    ('ΔSAR vs ΔPIB',d_sar,d_gdp),('ΔSAR vs ΔPop',d_sar,d_urb)]:
    x=np.asarray(x,dtype=float); y=np.asarray(y,dtype=float)
    mask = ~(np.isnan(x)|np.isnan(y)); r,p = stats.pearsonr(x[mask],y[mask])
    if label=='ΔSAR vs ΔICC': r_d, p_d = r, p
    print(f'  {label:28s}  r={r:+.3f}  p={p:.4f}  → {"SIGNIF ***" if p<0.05 else "non signif."}')

print('\nÉTAPE 4 — COINTÉGRATION ENGLE-GRANGER')
print('='*60)
_s = sar[~np.isnan(sar)]; _i = icc[~np.isnan(icc)]
_n = min(len(_s), len(_i))
score, p_coint, _ = coint(_s[:_n], _i[:_n])
print(f'  stat={score:.4f}  p={p_coint:.4f}  → {"COINTÉGRÉES ✓" if p_coint<0.05 else "NON cointégrées"}')
print(f'  ⚠️  n={_n} observations → faible puissance statistique')

print('\nÉTAPE 5 — CAUSALITÉ DE GRANGER (ΔICC → ΔSAR)')
print('='*60)
m = min(len(d_sar), len(d_icc))
data_gc = np.column_stack([d_sar[:m], d_icc[:m]])
data_gc = data_gc[~np.isnan(data_gc).any(axis=1)]
gc_annual = {}
for lag, res in grangercausalitytests(data_gc, maxlag=2, verbose=False).items():
    f_stat, p_val = res[0]['ssr_ftest'][0], res[0]['ssr_ftest'][1]
    gc_annual[lag] = p_val
    print(f'  Lag {lag}: F={f_stat:.3f}  p={p_val:.4f}  → '
          f'{"Granger-cause ✓" if p_val<0.05 else "pas de causalité Granger"}')
p_gc_annual = gc_annual.get(1, np.nan)

print('\n⚠️  ICC CORRIGÉ : ces résultats REMPLACENT ceux de la note de synthèse,')
print('    qui reposaient sur une série ICC décalée de 2 trimestres (2T2020+).')

In [ ]:
# ── CELL 24 — Visualisation Tests de Robustesse ─────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Tests de Stationnarité — Robustesse de la Relation ICC ↔ SAR\n'
             f'Nouakchott {eco_years[0]}-{eco_years[-1]}  ·  ICC corrigé  ·  n={len(sar)}',
             fontsize=13, fontweight='bold')

ax=axes[0,0]; ax2=ax.twinx()
ax.plot(eco_years, sar, color='#1d3557', lw=2.5, marker='D', ms=5, label='SAR (gauche)')
ax2.plot(eco_years, icc, color='#e63946', lw=2.5, marker='s', ms=5, ls='--', label='ICC (droite)')
ax.set_title(f'Niveaux — Tendances communes\np(ADF) SAR={adf_lvl["SAR index"]:.2f} · ICC={adf_lvl["ICC global"]:.2f}',
             fontweight='bold', fontsize=9)
ax.set_ylabel('SAR',color='#1d3557',fontsize=8); ax2.set_ylabel('ICC',color='#e63946',fontsize=8)
lines=ax.get_lines()+ax2.get_lines(); ax.legend(lines,[l.get_label() for l in lines],fontsize=8)
ax.set_xticks(eco_years); ax.set_xticklabels([str(y) for y in eco_years],rotation=45,fontsize=7); ax.grid(alpha=0.2,ls=':')

ax=axes[0,1]; ax2=ax.twinx()
ax.plot(yrs_d, d_sar, color='#1d3557', lw=2, marker='D', ms=5, label='ΔSAR')
ax2.plot(yrs_d, d_icc, color='#e63946', lw=2, marker='s', ms=5, ls='--', label='ΔICC')
ax.axhline(0,color='black',lw=0.8,alpha=0.5); ax2.axhline(0,color='black',lw=0.8,alpha=0.5)
ax.set_title('Différences premières\nTendances supprimées', fontweight='bold', fontsize=9)
ax.set_ylabel('ΔSAR',color='#1d3557',fontsize=8); ax2.set_ylabel('ΔICC',color='#e63946',fontsize=8)
lines=ax.get_lines()+ax2.get_lines(); ax.legend(lines,[l.get_label() for l in lines],fontsize=8)
ax.set_xticks(yrs_d); ax.set_xticklabels([str(y) for y in yrs_d],rotation=45,fontsize=7); ax.grid(alpha=0.2,ls=':')

ax=axes[0,2]
ax.scatter(d_icc, d_sar, color='#e63946', s=80, zorder=5, alpha=0.85)
for i,y in enumerate(yrs_d): ax.annotate(str(y),(d_icc[i],d_sar[i]),xytext=(3,3),textcoords='offset points',fontsize=7.5)
z_fit=np.polyfit(d_icc,d_sar,1); xs=np.linspace(d_icc.min(),d_icc.max(),100)
ax.plot(xs,np.poly1d(z_fit)(xs),color='#1d3557',lw=2,ls='--')
ax.set_xlabel('ΔICC'); ax.set_ylabel('ΔSAR')
ax.set_title(f'Scatter sur différences\nr={r_d:.3f}  p={p_d:.3f}', fontweight='bold', fontsize=9)
ax.axhline(0,color='black',lw=0.5,alpha=0.5); ax.axvline(0,color='black',lw=0.5,alpha=0.5); ax.grid(alpha=0.2,ls=':')

corr_diff = lag_correlation(d_icc, d_sar, max_lag=2)
ax=axes[1,0]
lgs=corr_diff.index.tolist(); rs=corr_diff['r'].tolist(); ps=corr_diff['p'].tolist()
bcolors=['#e63946' if (not pd.isna(r) and p<0.05) else '#f4a261' if (not pd.isna(r) and p<0.10) else '#dddddd' for r,p in zip(rs,ps)]
bars=ax.bar(lgs,rs,color=bcolors,alpha=0.85,width=0.5)
ax.axhline(0,color='black',lw=0.8)
for bar,r,p in zip(bars,rs,ps):
    if pd.isna(r): continue
    star='***' if p<0.01 else '**' if p<0.05 else '*' if p<0.10 else ''
    if star: ax.text(bar.get_x()+bar.get_width()/2,r+(0.03 if r>=0 else -0.07),star,ha='center',fontsize=10,fontweight='bold')
ax.set_title('Corrélations décalées ΔICC vs ΔSAR', fontweight='bold', fontsize=9)
ax.set_xlabel('Décalage',fontsize=8); ax.set_ylabel('r de Pearson',fontsize=8)
ax.set_ylim(-1.05,1.05); ax.set_xticks(lgs); ax.set_xticklabels([f'{l:+d}' for l in lgs],fontsize=8); ax.grid(axis='y',alpha=0.25,ls=':')

ax=axes[1,1]
def norm01(x):
    x=np.asarray(x,dtype=float); return (x-np.nanmin(x))/(np.nanmax(x)-np.nanmin(x)+1e-10)
ax.plot(yrs_d, norm01(d_sar), color='#1d3557',lw=2,marker='D',ms=5,label='ΔSAR')
ax.plot(yrs_d, norm01(d_icc), color='#e63946',lw=2,marker='s',ms=4,ls='--',label='ΔICC global')
ax.plot(yrs_d, norm01(d_lab), color='#f4a261',lw=2,marker='^',ms=4,ls=':',label='ΔICC main-œuvre')
ax.plot(yrs_d, norm01(d_gdp), color='#2a9d8f',lw=2,marker='v',ms=4,ls='-.',label='ΔPIB growth')
ax.set_title('Séries différenciées normalisées (0-1)', fontweight='bold', fontsize=9)
ax.set_ylabel('Valeur normalisée',fontsize=8); ax.legend(fontsize=7.5,loc='upper left')
ax.set_xticks(yrs_d); ax.set_xticklabels([str(y) for y in yrs_d],rotation=45,fontsize=7); ax.grid(alpha=0.2,ls=':')

# Tableau résumé — DYNAMIQUE (plus de valeurs codées en dur)
ax=axes[1,2]; ax.axis('off')
def verdict(p, thr=0.05, yes='Stat. ✓', no='Non-stat.'):
    return yes if p < thr else no
rows = [
    ['Test','Résultat','Interprétation'],
    ['ADF SAR (niveaux)',  f'p={adf_lvl["SAR index"]:.3f}',  verdict(adf_lvl["SAR index"],0.05,'Stationnaire','Tendancielle')],
    ['ADF ICC (niveaux)',  f'p={adf_lvl["ICC global"]:.3f}', verdict(adf_lvl["ICC global"],0.05,'Stationnaire','Tendancielle')],
    ['ADF ΔSAR',           f'p={adf_dif["ΔSAR"]:.3f}',       verdict(adf_dif["ΔSAR"],0.05,'I(1)','I(2)?')],
    ['ADF ΔICC',           f'p={adf_dif["ΔICC"]:.3f}',       verdict(adf_dif["ΔICC"],0.05,'I(1)','I(2)?')],
    ['Cointégration EG',   f'p={p_coint:.3f}',               f'n={len(sar)} → faible puiss.'],
    ['r(ΔSAR,ΔICC)',       f'r={r_d:+.3f} (p={p_d:.2f})',    verdict(p_d,0.05,'Co-mouvement','Pas de co-mouv.')],
    ['Granger ΔICC→ΔSAR',  f'p={p_gc_annual:.3f}',           verdict(p_gc_annual,0.05,'Causalité ✓','Pas de causalité')],
]
tbl = ax.table(cellText=rows[1:],colLabels=rows[0],cellLoc='center',loc='center',bbox=[0,0,1,1])
tbl.auto_set_font_size(False); tbl.set_fontsize(8.5)
for (r,c),cell in tbl.get_celld().items():
    if r==0: cell.set_facecolor('#1d3557'); cell.set_text_props(color='white',fontweight='bold')
    elif r%2==0: cell.set_facecolor('#f0f4f8')
ax.set_title(f'Résumé des Tests (annuel, n={len(sar)})', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/idx_05_stationarity_tests.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: idx_05_stationarity_tests.png')

---
## ✅ Récapitulatif intermédiaire (analyse annuelle 2016-2024)

> ⚠️ **Résultats intermédiaires.** Ce bilan porte sur l'analyse annuelle (n = 9 années).
> Les résultats finaux — série étendue à 2026 T2, indice trimestriel désaisonnalisé,
> cointégration de Johansen et VECM sur n = 41 trimestres — sont dans les Parties 11 et 12
> et résumés dans le `README.md` du dépôt.

### Résultats principaux

**1. Expansion urbaine confirmée (2016→2024)**
- Croissance moyenne ville : **+9.8 pp** sur 8 ans (+1.22 pp/an)
- Zones les plus dynamiques : **Ksar (+16.8 pp)**, **Teyarett (+14.9 pp)**, **Sebkha (+13.3 pp)**
- Zones saturées/stables : **Arafat** (97% de fraction urbaine)

**2. Indice composite — Classement**

| Rang | Zone | Indice |
|:----:|------|:------:|
| 1 | Ksar | 57.5 |
| 2 | Dar Naim | 55.4 |
| 3 | Sebkha | 50.0 |
| 4 | Arafat | 50.0 |
| 5 | Teyarett | 48.6 |

**3. Indice synthétique ville**  
Progression de **33.5 → 40.5 points** (+7.0 pts, +0.87 pts/an)

**4. Relation économique**
- ICC et SAR sont **co-intégrés** mais sans causalité de Granger
- Tous deux sont des manifestations du même **boom de construction**
- La corrélation en niveaux (r=0.97) est **partiellement fallacieuse** (séries non-stationnaires)

---
### Fichiers produits

| Fichier | Description |
|---------|-------------|
| `gt_01_spectral_profiles.png` | Profils spectraux S2 aux points terrain |
| `gt_03_delta_distribution.png` | Distribution ΔSWIR1 — diagnostic optique |
| `gt_04_sar_urban_map.png` | Carte SAR principale (4 panneaux) |
| `gt_04b_sar_evolution_annuelle.png` | VV annuel 2016-2024 + masques |
| `gt_04c_sar_2016_vs_2024.png` | Comparaison directe + ΔBackscatter |
| `gt_04d_sar_zoom_fronts.png` | Zoom fronts urbanisation (Dar Naim, Toujounine, Ksar) |
| `gt_05_sar_bar_charts.png` | Métriques SAR par moughataa |
| `gt_06_timeseries_panels.png` | Série temporelle — 9 panneaux individuels |
| `gt_07_timeseries_combined.png` | Série temporelle — toutes zones |
| `idx_01_choropleth.png` | Cartes choroplèthes de l'indice |
| `idx_02_index_breakdown.png` | Décomposition + évolution annuelle |
| `idx_03_city_synthetic_index.png` | Indice synthétique ville |
| `idx_04_correlation_analysis.png` | Corrélations + matrice |
| `idx_05_stationarity_tests.png` | Tests économétriques |
| `sar_urban_fraction_timeseries.csv` | Données séries temporelles |

---
## 📅 PARTIE 11 — Décomposition Mensuelle de l'Indice par Moughataa et Global

### Objectif
Passer de l'analyse **annuelle** à l'analyse **mensuelle** pour :
- Détecter une **saisonnalité** dans le signal SAR (saison des pluies juil-sept vs saison sèche)
- Valider que la **tendance long-terme** reste robuste après décomposition saisonnière
- Identifier quelles moughataat réagissent le plus aux variations climatiques saisonnières

### Approche
On travaille sur le **VV moyen continu (dB)** — pas de seuillage binaire à -14 dB.  
Avec seulement 2-3 images par mois, le seuillage est trop instable.  
La valeur VV moyenne continue absorbe mieux le bruit inter-image.

### Hypothèses
| # | Hypothèse | Attendu |
|---|-----------|---------|
| H1 | Saisonnalité positive juil-sept | +0.5 à +1.5 dB zones périphériques |
| H2 | Zones très urbanisées = faible saisonnalité | Ksar, Arafat < Toujounine, Riyadh |
| H3 | Tendance long-terme visible après décomposition | Valide l'indice annuel |
| H4 | Résiduel dominé par bruit SAR (2-3 img/mois) | Résiduel ±1 dB aléatoire |

### 📥 Cellule A — Extraction VV Mensuel 2016-2024
> ⏱️ **Attention : cette cellule prend ~20-30 minutes.**  
> Elle fait 972 appels GEE (9 zones × 12 mois × 9 ans).  
> Lancer et ne pas interrompre. Le résultat est sauvegardé en CSV.

In [ ]:
# ── CELL 11A — Extraction Mensuelle par Moughataa (extracteur générique) ────
import os
CACHE = '../outputs/monthly_sar_vv.csv'

if os.path.exists(CACHE):
    df_monthly = pd.read_csv(CACHE)
    print(f'✓ Rechargé depuis le cache : {len(df_monthly)} lignes (aucun appel GEE)')
else:
    df_monthly = extract_monthly(s1_col, 'VV', 10, value_name='vv_mean', label='Sentinel-1 VV')
    df_monthly.to_csv(CACHE, index=False)
    print('✓ Extrait de GEE et mis en cache')

import calendar, time
from datetime import date

MONTHS      = list(range(1, 13))
MONTH_NAMES = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']

# FeatureCollection des 9 moughataat — construite UNE fois
ZONES_FC = ee.FeatureCollection([
    ee.Feature(
        ee.Geometry(_json.loads(
            gpd.GeoSeries([row.geometry], crs='EPSG:4326').to_json())['features'][0]['geometry']),
        {'zone': row['ADM2_EN']}
    )
    for _, row in gdf.iterrows()
])

def extract_monthly(collection, band, scale, value_name='value',
                    year_start=YEAR_START, last=MONTHLY_LAST,
                    reducer=ee.Reducer.mean(), label=''):
    """Moyenne zonale mensuelle d'une collection GEE, par moughataa.
       Réutilisable : Sentinel-1 (10 m), VIIRS (~500 m), CHIRPS (~5 km)."""
    records, t0 = [], time.time()
    zones = list(gdf['ADM2_EN'])
    months_todo = [(y, m) for y in range(year_start, last.year + 1) for m in MONTHS
                   if date(y, m, 1) <= last]
    print(f'{label} — {len(months_todo)} mois × {len(zones)} zones '
          f'({year_start}-01 → {last:%Y-%m}) | scale={scale} m')

    for k, (year, month) in enumerate(months_todo, 1):
        d1 = f'{year}-{month:02d}-01'
        d2 = f'{year}-{month:02d}-{calendar.monthrange(year, month)[1]}'
        col_m = collection.filterDate(d1, d2).select(band)
        n_img = col_m.size().getInfo()

        if n_img == 0:                                   # mois sans image → NaN
            records += [{'year':year,'month':month,'zone':z,
                         value_name:None,'n_images':0} for z in zones]
        else:
            fc = col_m.median().reduceRegions(           # ← 9 zones en 1 appel
                collection=ZONES_FC, reducer=reducer, scale=scale)
            for f in fc.getInfo()['features']:
                p = f['properties']
                v = p.get('mean', p.get(band))
                records.append({'year':year,'month':month,'zone':p['zone'],
                                value_name: round(float(v), 3) if v is not None else None,
                                'n_images': n_img})

        if k % 20 == 0 or k == len(months_todo):
            el = time.time() - t0
            print(f'  {k:3d}/{len(months_todo)}  {year}-{month:02d}  '
                  f'n_img={n_img:2d}  ETA {el/k*(len(months_todo)-k)/60:4.1f} min')

    df = pd.DataFrame(records)
    print(f'✓ {label} : {len(df)} lignes en {(time.time()-t0)/60:.1f} min  '
          f'| NaN : {df[value_name].isna().sum()} '
          f'({df[value_name].isna().mean()*100:.1f}%)\n')
    return df

# ── Extraction Sentinel-1 VV ───────────────────────────────────────────────
df_monthly = extract_monthly(s1_col, 'VV', 10, value_name='vv_mean', label='Sentinel-1 VV')
df_monthly.to_csv('../outputs/monthly_sar_vv.csv', index=False)
print('✓ Saved: ../outputs/monthly_sar_vv.csv')

# ── Sous-ensemble « années complètes » : pour TOUTE analyse de saisonnalité ─
# (compare des mois entre eux → il faut le même nombre d'années par mois)
df_monthly_full = df_monthly[df_monthly['year'] <= ANNUAL_END].copy()
print(f'✓ df_monthly_full : {YEAR_START}-{ANNUAL_END} '
      f'({df_monthly_full["year"].nunique()} années complètes) → 11D/11E/11G/11H')
print(f'  df_monthly      : jusqu\'à {MONTHLY_END} (2026 partiel) → 11F (STL) uniquement')

### 🔍 Cellule B — Qualité des Données Mensuelles
Avant toute analyse : vérifier la couverture temporelle, le nombre d'images, les NaN.

In [ ]:
# ── CELL 11B — Qualité et Couverture des Données Mensuelles ────────────────
# Si tu relances après avoir fermé le notebook :
df_monthly = pd.read_csv('../outputs/monthly_sar_vv.csv')

print('── Statistiques de couverture ──────────────────────────────────────────')
print(f'  Enregistrements totaux : {len(df_monthly)}')
print(f'  Zones                  : {df_monthly["zone"].nunique()}')
print(f'  Mois-années            : {df_monthly.groupby(["year","month"]).ngroups}')
print(f'  NaN (0 image ce mois)  : {df_monthly["vv_mean"].isna().sum()}')
print()

# Images par mois (moyenne sur toutes les zones)
img_per_month = (df_monthly.groupby(['year','month'])['n_images']
                 .first().unstack('month'))
print('── Nombre d\'images Sentinel-1 par mois ───────────────────────────────')
print(f'{"":6s}', end='')
for m in MONTHS: print(f'{MONTH_NAMES[m-1]:>6s}', end='')
print()
print('-'*(6+6*12))
for year in YEARS_MONTHLY:
    print(f'{year}  ', end='')
    for month in MONTHS:
        if date(year, month, 1) > MONTHLY_LAST:
            print(f'{"·":>6s}', end='')          # hors période, PAS zéro image
            continue
        n = (img_per_month.loc[year, month]
             if (year in img_per_month.index and month in img_per_month.columns) else 0)
        print(f'{int(n):>6d}', end='')
    print()

print(f'\n  ⚠️  2016 : {int(img_per_month.loc[2016].sum())} images seulement (S1A seul, montée en charge)')
print(f'  ⚠️  2022+ : ~24/an — panne de Sentinel-1B (déc. 2021)')

print()
print(f'  Moyenne images/mois : {df_monthly.groupby(["year","month"])["n_images"].first().mean():.1f}')
print(f'  Min                 : {df_monthly.groupby(["year","month"])["n_images"].first().min()}')
print(f'  Mois avec 0 image   : {(df_monthly.groupby(["year","month"])["n_images"].first()==0).sum()}')
print()

# Plage VV par zone
print('── Plage VV moyen par moughataa ─────────────────────────────────────')
print(f'{"Zone":20s} {"Min":>8s} {"Max":>8s} {"Moy":>8s} {"Std":>8s}')
print('-'*55)
for zone in ZONE_ORDER:
    sub = df_monthly[df_monthly['zone']==zone]['vv_mean'].dropna()
    print(f'{zone:20s} {sub.min():>8.2f} {sub.max():>8.2f} {sub.mean():>8.2f} {sub.std():>8.2f}')

### 🗓️ Cellule C — Heatmap Brute Année × Mois
Première visualisation : voir la structure temporelle globale du signal VV.

In [ ]:
# ── CELL 11C — Heatmap VV Mensuel par Moughataa ─────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(20, 16))
fig.suptitle('VV Backscatter Mensuel (dB) — Nouakchott 2016-2024\n'
             'Heatmap Année × Mois par Moughataa  |  Blanc=fort=urbain  |  Bleu=faible=sable',
             fontsize=14, fontweight='bold')

cmap_vv = LinearSegmentedColormap.from_list('vv_heat',
    ['#2c7bb6','#abd9e9','#ffffbf','#fdae61','#d7191c'])

for ax, zone in zip(axes.flat, ZONE_ORDER):
    sub = df_monthly[df_monthly['zone']==zone].copy()
    pivot = sub.pivot(index='year', columns='month', values='vv_mean')

    # Valeurs min/max propres à la zone pour contraste max
    vmin = pivot.values[~np.isnan(pivot.values)].min()
    vmax = pivot.values[~np.isnan(pivot.values)].max()

    im = ax.imshow(pivot.values, aspect='auto', cmap=cmap_vv,
                   vmin=vmin, vmax=vmax,
                   interpolation='nearest')

    # Annotations valeurs
    for i, year in enumerate(YEARS_MONTHLY):
        for j, month in enumerate(MONTHS):
            val = pivot.loc[year, month] if (year in pivot.index and month in pivot.columns) else np.nan
            if not np.isnan(val):
                ax.text(j, i, f'{val:.1f}', ha='center', va='center',
                        fontsize=5.5, color='black' if (val-vmin)/(vmax-vmin+1e-10)>0.4 else 'white')

    ax.set_xticks(range(12))
    ax.set_xticklabels(MONTH_NAMES, fontsize=7, rotation=45)
    ax.set_yticks(range(len(YEARS_MONTHLY)))
    ax.set_yticklabels([str(y) for y in YEARS_MONTHLY], fontsize=7)
    ax.set_title(zone, fontsize=10, fontweight='bold', color=ZONE_COLORS[zone])
    plt.colorbar(im, ax=ax, shrink=0.7, pad=0.02, label='VV (dB)')

plt.tight_layout()
plt.savefig('../outputs/monthly_01_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: monthly_01_heatmap.png')

# Heatmap ville (moyenne pondérée superficie)
fig, ax = plt.subplots(figsize=(16, 5))
fig.suptitle('VV Backscatter Mensuel — Indice Ville Global (pondéré superficie)\n'
             'Nouakchott 2016-2024', fontsize=13, fontweight='bold')

# Calculer la moyenne pondérée ville par mois-année
city_monthly = []
for year in YEARS_MONTHLY:
    for month in MONTHS:
        sub = df_monthly[(df_monthly['year']==year) & (df_monthly['month']==month)]
        vv_w = sum(sub[sub['zone']==z]['vv_mean'].values[0] * areas[z]
                   for z in ZONE_ORDER
                   if len(sub[sub['zone']==z])>0 and sub[sub['zone']==z]['vv_mean'].values[0] is not None
                   and not np.isnan(sub[sub['zone']==z]['vv_mean'].values[0]))
        w_total = sum(areas[z] for z in ZONE_ORDER
                      if len(sub[sub['zone']==z])>0 and sub[sub['zone']==z]['vv_mean'].values[0] is not None
                      and not np.isnan(sub[sub['zone']==z]['vv_mean'].values[0]))
        city_monthly.append({'year':year,'month':month,
                             'vv_city': round(vv_w/w_total,3) if w_total>0 else np.nan})

df_city_monthly = pd.DataFrame(city_monthly)
pivot_city = df_city_monthly.pivot(index='year', columns='month', values='vv_city')

vmin_c = pivot_city.values[~np.isnan(pivot_city.values)].min()
vmax_c = pivot_city.values[~np.isnan(pivot_city.values)].max()

im = ax.imshow(pivot_city.values, aspect='auto', cmap=cmap_vv,
               vmin=vmin_c, vmax=vmax_c)
for i, year in enumerate(YEARS_MONTHLY):
    for j, month in enumerate(MONTHS):
        val = pivot_city.loc[year, month] if year in pivot_city.index else np.nan
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7,
                    color='black' if (val-vmin_c)/(vmax_c-vmin_c+1e-10)>0.4 else 'white')
ax.set_xticks(range(12))
ax.set_xticklabels(MONTH_NAMES, fontsize=9)
ax.set_yticks(range(len(YEARS_MONTHLY)))
ax.set_yticklabels([str(y) for y in YEARS_MONTHLY], fontsize=9)
ax.set_title('Ville de Nouakchott — Indice Global', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.6, label='VV moyen pondéré (dB)')
plt.tight_layout()
plt.savefig('../outputs/monthly_01b_heatmap_city.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: monthly_01b_heatmap_city.png')

### 🗺️ Cellule D — Cartes SAR Saison Sèche vs Saison des Pluies
Visualisation spatiale de la différence entre les deux saisons.

In [ ]:
# ── CELL 11D — Cartes SAR : Saison Sèche vs Saison des Pluies ──────────────
# Composite de TOUS les mois de janvier (saison sèche) vs TOUS les août (pluies)
# sur la période 2016-2024 → composite stable malgré peu d'images/mois

print('Construction composites saisonniers multi-annuels...')

# Tous les janvier 2016-2024 → composite saison sèche stable
# Années complètes UNIQUEMENT : sinon janvier aurait 11 années et août 10
# → composite « sèche » gonflé par un janvier 2026 récent (série haussière)
SEASON_END = f'{ANNUAL_END}-12-31'
s1_seas = s1_col.filterDate(f'{YEAR_START}-01-01', SEASON_END)
print(f'Composites saisonniers sur {YEAR_START}-{ANNUAL_END} (années complètes)')

col_jan = s1_seas.filter(ee.Filter.calendarRange(1, 1, 'month')).select('VV')
col_aug = s1_seas.filter(ee.Filter.calendarRange(8, 8, 'month')).select('VV')
col_dry = s1_seas.filter(ee.Filter.Or(
    ee.Filter.calendarRange(11, 12, 'month'),
    ee.Filter.calendarRange(1,  4,  'month'))).select('VV')
col_wet = s1_seas.filter(ee.Filter.calendarRange(7, 10, 'month')).select('VV')

n_jan = col_jan.size().getInfo()
n_aug = col_aug.size().getInfo()
n_dry = col_dry.size().getInfo()
n_wet = col_wet.size().getInfo()
print(f'  Janvier (multi-an) : {n_jan} images')
print(f'  Août    (multi-an) : {n_aug} images')
print(f'  Saison sèche totale: {n_dry} images')
print(f'  Saison pluies total: {n_wet} images')

comp_jan = col_jan.median()
comp_aug = col_aug.median()
comp_dry = col_dry.median()
comp_wet = col_wet.median()

# Différence pluies - sèche
vv_diff_seasonal = comp_wet.subtract(comp_dry).rename('vv_diff')

print('Téléchargement thumbnails saisonniers...')
vv_dry_thumb  = get_colored_thumb(comp_dry, 'VV', region_ee, PALETTE_VV, -20, -8)
vv_wet_thumb  = get_colored_thumb(comp_wet, 'VV', region_ee, PALETTE_VV, -20, -8)
vv_jan_thumb  = get_colored_thumb(comp_jan, 'VV', region_ee, PALETTE_VV, -20, -8)
vv_aug_thumb  = get_colored_thumb(comp_aug, 'VV', region_ee, PALETTE_VV, -20, -8)
vv_diff_thumb = get_colored_thumb(vv_diff_seasonal, 'vv_diff', region_ee,
                    ['2166ac','4393c3','f7f7f7','f4a582','d6604d'], -2, 2)
print('✓ OK')

fig, axes = plt.subplots(1, 5, figsize=(30, 9))
fig.suptitle('Comparaison Saisonnière SAR — Saison Sèche vs Saison des Pluies\n'
             'Composites multi-annuels 2016-2024  |  Sentinel-1 VV (dB)',
             fontsize=13, fontweight='bold')

for ax, (img, title) in zip(axes, [
    (vv_jan_thumb,  'Composite Janvier\n(saison sèche — tous les jan 2016-2024)'),
    (vv_dry_thumb,  'Composite Saison Sèche\n(Nov-Avr, multi-annuel)'),
    (vv_aug_thumb,  'Composite Août\n(saison pluies — tous les aoû 2016-2024)'),
    (vv_wet_thumb,  'Composite Saison Pluies\n(Jul-Oct, multi-annuel)'),
    (vv_diff_thumb, 'ΔVV = Saison Pluies − Sèche\nRouge=hausse | Bleu=baisse'),
]):
    ax.imshow(img, extent=[minx, maxx, miny, maxy], origin='upper')
    ax.set_aspect('equal')
    annotate_map(ax, gdf, bounds)
    plot_gt(ax)
    ax.set_title(title, fontsize=8.5, fontweight='bold')

# Annotation sur la carte diff
axes[4].text(0.02, 0.04,
    'Rouge = zones qui\nréagissent aux pluies\n(sol nu, végétation)\nBleu = béton/routes',
    transform=axes[4].transAxes, fontsize=8, color='white',
    bbox=dict(facecolor='black', alpha=0.7, boxstyle='round'))

plt.tight_layout()
plt.savefig('../outputs/monthly_02_maps_seasonal.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: monthly_02_maps_seasonal.png')

# Stats ΔVV par zone (saison pluies - sèche)
print('\nΔVV par moughataa (saison pluies - saison sèche) :')
print(f'{"Zone":20s} {"VV_sèche":>10s} {"VV_pluies":>10s} {"ΔVV":>8s}  Saisonnalité')
print('-'*60)
for _, row in gdf.iterrows():
    zone    = row['ADM2_EN']
    geom_ee = ee.Geometry(_json.loads(gpd.GeoSeries([row.geometry],crs='EPSG:4326').to_json())['features'][0]['geometry'])
    vv_d = comp_dry.reduceRegion(ee.Reducer.mean(),geom_ee,10,maxPixels=1e10,bestEffort=True).getInfo().get('VV',np.nan)
    vv_w = comp_wet.reduceRegion(ee.Reducer.mean(),geom_ee,10,maxPixels=1e10,bestEffort=True).getInfo().get('VV',np.nan)
    if vv_d and vv_w:
        diff = float(vv_w)-float(vv_d)
        flag = 'Forte ↑' if diff>0.5 else 'Modérée' if diff>0.2 else 'Faible'
        print(f'{zone:20s} {float(vv_d):>10.2f} {float(vv_w):>10.2f} {diff:>+8.2f}  {flag}')

### 📦 Cellule E — Boxplot Saisonnalité par Mois
Distribution du VV pour chaque mois, poolé sur toutes les années.  
Révèle le **profil saisonnier moyen** de chaque moughataa.

In [ ]:
# ── CELL 11E — Boxplot Saisonnalité Mensuelle ───────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(20, 16))
fig.suptitle('Saisonnalité du Backscatter VV — Distribution par Mois (2016-2024)\n'
             'Chaque boîte = distribution de ce mois sur toutes les années disponibles',
             fontsize=14, fontweight='bold')

for ax, zone in zip(axes.flat, ZONE_ORDER):
    sub   = df_monthly[df_monthly['zone']==zone].dropna(subset=['vv_mean'])
    color = ZONE_COLORS[zone]

    # Boxplot par mois
    data_by_month = [sub[sub['month']==m]['vv_mean'].values for m in MONTHS]
    bp = ax.boxplot(data_by_month, patch_artist=True, notch=False,
                    medianprops=dict(color='black', lw=2),
                    whiskerprops=dict(color=color, lw=1.2),
                    capprops=dict(color=color, lw=1.5),
                    flierprops=dict(marker='o', markerfacecolor=color, ms=3, alpha=0.5))
    for patch in bp['boxes']:
        patch.set_facecolor(color); patch.set_alpha(0.35)

    # Profil des médianes
    medians = [np.median(d) if len(d)>0 else np.nan for d in data_by_month]
    ax2 = ax.twinx()
    ax2.plot(range(1, 13), medians, color=color, lw=2.5, marker='o', ms=5, zorder=5)
    ax2.set_ylabel('Médiane VV (dB)', fontsize=7, color=color)
    ax2.tick_params(axis='y', labelcolor=color, labelsize=7)

    # Zones de saison
    ax.axvspan(7, 10, alpha=0.08, color='blue', label='Saison pluies')
    ax.axvspan(1,  4, alpha=0.08, color='orange', label='Saison sèche')

    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_NAMES, fontsize=7, rotation=45)
    ax.set_ylabel('VV (dB)', fontsize=8)
    ax.set_title(zone, fontsize=10, fontweight='bold', color=color)
    ax.grid(alpha=0.2, ls=':')

# Légende commune
axes[0,0].axvspan(0,0, alpha=0.3, color='blue',   label='Saison pluies (Jul-Oct)')
axes[0,0].axvspan(0,0, alpha=0.3, color='orange', label='Saison sèche (Nov-Avr)')
axes[0,0].legend(fontsize=7, loc='upper left')

plt.tight_layout()
plt.savefig('../outputs/monthly_03_boxplot_seasonality.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: monthly_03_boxplot_seasonality.png')

# Profil moyen ville
fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('Profil Saisonnier Moyen — Ville de Nouakchott (2016-2024)\n'
             'VV mensuel poolé sur toutes les années', fontsize=12, fontweight='bold')

for zone in ZONE_ORDER:
    sub     = df_monthly_full[df_monthly_full['zone']==zone] .dropna(subset=['vv_mean'])
    medians = [np.median(sub[sub['month']==m]['vv_mean'].values) if len(sub[sub['month']==m])>0 else np.nan
               for m in MONTHS]
    ax.plot(range(1,13), medians, color=ZONE_COLORS[zone], lw=1.5,
            marker='o', ms=4, alpha=0.7, label=zone)

ax.axvspan(7, 10, alpha=0.08, color='blue',   label='Saison pluies')
ax.axvspan(11,12, alpha=0.06, color='orange')
ax.axvspan(1,  4, alpha=0.06, color='orange', label='Saison sèche')
ax.set_xticks(range(1,13)); ax.set_xticklabels(MONTH_NAMES, fontsize=9)
ax.set_ylabel('VV médian (dB)'); ax.set_xlabel('Mois')
ax.legend(fontsize=8, loc='upper right', ncol=2); ax.grid(alpha=0.25, ls=':')
plt.tight_layout()
plt.savefig('../outputs/monthly_03b_seasonal_profile_city.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: monthly_03b_seasonal_profile_city.png')

### 📉 Cellule F — Décomposition STL (Tendance + Saisonnier + Résiduel)
Décomposition classique de série temporelle en trois composantes :
- **Tendance** — la croissance urbaine long-terme
- **Saisonnalité** — le cycle annuel répétitif
- **Résiduel** — le bruit restant (bruit SAR, événements rares)

In [ ]:
# ── CELL 11F — Décomposition STL par Moughataa ──────────────────────────────
from statsmodels.tsa.seasonal import STL

fig, axes = plt.subplots(9, 4, figsize=(22, 36))
fig.suptitle('Décomposition STL — VV Mensuel par Moughataa 2016-2024\n'
             'Observed · Trend · Seasonal · Residual',
             fontsize=14, fontweight='bold')

stl_results = {}

for row_i, zone in enumerate(ZONE_ORDER):
    sub = (df_monthly_full[df_monthly_full['zone']==zone]
           .sort_values(['year','month'])
           .dropna(subset=['vv_mean'])
           .copy())

    # Créer index mensuel complet
    sub['date'] = pd.to_datetime({'year':sub['year'],'month':sub['month'],'day':1})
    sub = sub.set_index('date')

    # Remplir les éventuels mois manquants par interpolation
    full_idx = pd.date_range(f'{YEAR_START}-01-01', f'{MONTHLY_END}-01', freq='MS')
    ts = sub['vv_mean'].reindex(full_idx).interpolate(method='linear')

    # STL : période = 12 mois
    stl   = STL(ts, period=12, robust=True)
    res   = stl.fit()
    stl_results[zone] = res

    color = ZONE_COLORS[zone]
    components = [
        (ts.values,           'Observé (VV brut)', color),
        (res.trend,           'Tendance',          '#1d3557'),
        (res.seasonal,        'Saisonnalité',      '#2a9d8f'),
        (res.resid,           'Résiduel',          '#adb5bd'),
    ]

    for col_i, (data, label, c) in enumerate(components):
        ax = axes[row_i, col_i]
        ax.plot(full_idx, data, color=c, lw=1.5 if col_i>0 else 1.0, alpha=0.9)
        if col_i == 0:
            ax.set_ylabel(zone, fontsize=8, fontweight='bold', color=color)
        if col_i == 2:  # saisonnalité : axhline à 0
            ax.axhline(0, color='black', lw=0.8, ls='--', alpha=0.5)
        if col_i == 3:  # résiduel : axhline ±1 dB
            ax.axhline(+1, color='red',  lw=0.8, ls=':', alpha=0.5)
            ax.axhline(-1, color='red',  lw=0.8, ls=':', alpha=0.5)
        if row_i == 0:
            ax.set_title(label, fontsize=9, fontweight='bold')
        ax.set_xticks([]); ax.grid(alpha=0.2, ls=':')
        ax.tick_params(labelsize=6)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('../outputs/monthly_04_stl_decomposition.png', dpi=120, bbox_inches='tight')
plt.close()
print('✓ Saved: monthly_04_stl_decomposition.png')

# Résumé des amplitudes
print('\nAmplitude saisonnière STL par zone (max-min de la composante saisonnière) :')
print(f'{"Zone":20s} {"Amplitude(dB)":>14s} {"Tendance slope":>16s} (dB/an)')
print('-'*55)
for zone in ZONE_ORDER:
    res   = stl_results[zone]
    amp   = res.seasonal.max() - res.seasonal.min()
    slope = np.polyfit(range(len(res.trend)), res.trend, 1)[0] * 12  # dB/an
    flag  = '↑ urbanisation' if slope > 0.05 else ('↓ ??' if slope < -0.05 else '→ stable')
    print(f'{zone:20s} {amp:>14.3f} {slope:>+16.3f}  {flag}')

### 📊 Cellule G — Test de Kruskal-Wallis (Significativité de la Saisonnalité)
Test non-paramétrique : est-ce que les 12 mois ont des distributions **significativement différentes** ?  
H₀ : toutes les distributions mensuelles sont identiques (pas de saisonnalité).

In [ ]:
# ── CELL 11G — Test Kruskal-Wallis Saisonnalité ─────────────────────────────
from scipy.stats import kruskal, f_oneway

print('Test de Kruskal-Wallis : H0 = pas de différence entre les mois')
print('p < 0.05 → saisonnalité statistiquement significative')
print('='*65)
print(f'{"Zone":20s} {"stat H":>10s} {"p-value":>10s} {"Signif.":>12s} {"Amp.(dB)":>10s}')
print('-'*65)

kw_results = {}
for zone in ZONE_ORDER:
    sub          = df_monthly[df_monthly['zone']==zone].dropna(subset=['vv_mean'])
    groups       = [sub[sub['month']==m]['vv_mean'].values for m in MONTHS]
    groups_clean = [g for g in groups if len(g)>=3]

    if len(groups_clean) < 6:
        print(f'{zone:20s}  {"N/A":>10s} {"N/A":>10s}  données insuffisantes')
        continue

    stat, p = kruskal(*groups_clean)
    amp      = max(np.median(g) for g in groups_clean) - min(np.median(g) for g in groups_clean)
    sig      = '*** p<0.001' if p<0.001 else '**  p<0.01' if p<0.01 else '*   p<0.05' if p<0.05 else 'ns  p>0.05'
    kw_results[zone] = {'stat':stat, 'p':p, 'amp':amp}
    print(f'{zone:20s} {stat:>10.3f} {p:>10.4f} {sig:>12s} {amp:>10.3f}')

print()
print('Interprétation :')
print('  Amplitude > 1.0 dB + p<0.05 → saisonnalité réelle et forte')
print('  Amplitude < 0.5 dB          → saisonnalité dominée par le bruit SAR')
print()

# Figure : p-values et amplitudes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Significativité de la Saisonnalité par Moughataa\n'
             'Test de Kruskal-Wallis sur les 12 distributions mensuelles',
             fontsize=12, fontweight='bold')

zones_kw = list(kw_results.keys())
pvals    = [kw_results[z]['p']   for z in zones_kw]
amps     = [kw_results[z]['amp'] for z in zones_kw]

ax = axes[0]
colors_p = ['#e63946' if p<0.05 else '#adb5bd' for p in pvals]
bars = ax.barh(zones_kw, [-np.log10(p) for p in pvals], color=colors_p, alpha=0.85)
ax.axvline(-np.log10(0.05), color='black', lw=1.5, ls='--', label='p=0.05')
ax.axvline(-np.log10(0.01), color='grey',  lw=1,   ls=':',  label='p=0.01')
ax.set_xlabel('-log10(p-value)  [plus grand = plus significatif]')
ax.set_title('Significativité (Kruskal-Wallis)', fontweight='bold')
ax.legend(fontsize=8); ax.grid(axis='x', alpha=0.3, ls=':')

ax = axes[1]
colors_a = ['#2a9d8f' if a>0.5 else '#adb5bd' for a in amps]
ax.barh(zones_kw, amps, color=colors_a, alpha=0.85)
ax.axvline(0.5, color='black', lw=1.5, ls='--', label='Seuil 0.5 dB')
ax.axvline(1.0, color='grey',  lw=1,   ls=':',  label='Seuil 1.0 dB')
ax.set_xlabel('Amplitude saisonnière (dB)  [médiane max - médiane min]')
ax.set_title('Amplitude Saisonnière', fontweight='bold')
ax.legend(fontsize=8); ax.grid(axis='x', alpha=0.3, ls=':')

plt.tight_layout()
plt.savefig('../outputs/monthly_04b_kruskal_wallis.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: monthly_04b_kruskal_wallis.png')

### 🗺️ Cellule H — Carte Différence Saisonnière Spatiale
Choroplèthe par moughataa : où la saisonnalité est-elle la plus forte ?  
**Hypothèse :** Les zones les plus urbanisées (béton imperméable) auront la **plus faible** amplitude — inverse logique de la carte d'urbanisation.

In [ ]:
# ── CELL 11H — Carte Choroplèthe Amplitude Saisonnière ──────────────────────
# Thumbnail de la différence VV août - janvier sur toute la ville

print('Téléchargement carte différence saisonnière...')
vv_aug_full = s1_col.filter(ee.Filter.calendarRange(8,8,'month')).select('VV').median()
vv_jan_full = s1_col.filter(ee.Filter.calendarRange(1,1,'month')).select('VV').median()
vv_aug_jan  = vv_aug_full.subtract(vv_jan_full).rename('seasonal_diff')

vv_aug_thumb2 = get_colored_thumb(vv_aug_full,'VV',region_ee,PALETTE_VV,-20,-8)
vv_jan_thumb2 = get_colored_thumb(vv_jan_full,'VV',region_ee,PALETTE_VV,-20,-8)
vv_diff_thumb2 = get_colored_thumb(vv_aug_jan,'seasonal_diff',region_ee,
                    ['2166ac','92c5de','f7f7f7','f4a582','d6604d'],-2,2)
print('✓ OK')

fig = plt.figure(figsize=(22, 12))
fig.suptitle('Saisonnalité Spatiale SAR — Amplitude Août vs Janvier\n'
             'Nouakchott 2016-2024 (composites multi-annuels)',
             fontsize=13, fontweight='bold')

gs = gridspec.GridSpec(1, 4, figure=fig, wspace=0.05)

# 3 cartes SAR
for col_i, (img, title) in enumerate([
    (vv_jan_thumb2,  'VV Janvier\n(saison sèche)'),
    (vv_aug_thumb2,  'VV Août\n(saison des pluies)'),
    (vv_diff_thumb2, 'ΔVV Août−Janvier\nRouge=+dB pluies | Bleu=−dB'),
]):
    ax = fig.add_subplot(gs[0, col_i])
    ax.imshow(img, extent=[minx,maxx,miny,maxy], origin='upper')
    ax.set_aspect('equal')
    annotate_map(ax, gdf, bounds)
    plot_gt(ax)
    ax.set_title(title, fontsize=9, fontweight='bold')

# Choroplèthe amplitude KW par moughataa
ax4 = fig.add_subplot(gs[0, 3])
gdf_seasonal = gdf.copy()
gdf_seasonal['amplitude'] = gdf_seasonal['ADM2_EN'].map(
    {z: kw_results[z]['amp'] if z in kw_results else np.nan for z in ZONE_ORDER}
)
gdf_seasonal['signif'] = gdf_seasonal['ADM2_EN'].map(
    {z: kw_results[z]['p']<0.05 if z in kw_results else False for z in ZONE_ORDER}
)

cmap_amp = LinearSegmentedColormap.from_list('amplitude',['#1d3557','#a8dadc','#e63946'])
gdf_seasonal.plot(column='amplitude', ax=ax4, cmap=cmap_amp,
                  vmin=0, vmax=gdf_seasonal['amplitude'].max(),
                  edgecolor='white', linewidth=1.5,
                  missing_kwds={'color':'lightgrey'})

for _, row in gdf_seasonal.iterrows():
    amp  = row['amplitude']
    sig  = row['signif']
    star = '*' if sig else ''
    if not np.isnan(amp):
        ax4.annotate(f"{row['ADM2_EN']}{amp:.2f}dB{star}",
                     (row.geometry.centroid.x, row.geometry.centroid.y),
                     fontsize=7.5, ha='center', va='center', fontweight='bold',
                     bbox=dict(boxstyle='round,pad=0.2',facecolor='white',alpha=0.75,linewidth=0))

sm = plt.cm.ScalarMappable(cmap=cmap_amp,
     norm=plt.Normalize(vmin=0, vmax=gdf_seasonal['amplitude'].max()))
sm.set_array([])
plt.colorbar(sm, ax=ax4, shrink=0.6, label='Amplitude saisonnière (dB)')
ax4.set_title('Amplitude Saisonnière par Moughataa\n(* = p<0.05 Kruskal-Wallis)',
              fontsize=9, fontweight='bold')
ax4.set_xticks([]); ax4.set_yticks([])

plt.savefig('../outputs/monthly_05_map_diff_saisons.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: monthly_05_map_diff_saisons.png')
print()
print('Corrélation amplitude saisonnière vs fraction urbaine 2024 :')
amps_ordered = [kw_results[z]['amp'] if z in kw_results else np.nan for z in ZONE_ORDER]
ufs_ordered  = [uf_2024[z] for z in ZONE_ORDER]
mask_corr    = ~np.isnan(amps_ordered)
from scipy.stats import pearsonr
r_amp, p_amp = pearsonr(np.array(amps_ordered)[mask_corr], np.array(ufs_ordered)[mask_corr])
print(f'  r = {r_amp:.3f}  p = {p_amp:.4f}')
print(f'  → {"Corrélation négative ✓ — plus urbanisé = moins saisonnier" if r_amp<0 else "Résultat inattendu"}')

### 📈 Cellule I — Synthèse Finale : Série Mensuelle Lissée
Figure publication-ready : série mensuelle brute + tendance lissée (moyenne mobile 12 mois) pour chaque moughataa et pour la ville.

In [ ]:
# ── CELL 11I — Synthèse Finale : Série Mensuelle + Tendance Lissée ──────────
fig, axes = plt.subplots(3, 3, figsize=(22, 16))
fig.suptitle('Série Mensuelle VV + Tendance Lissée (MM-12) — Nouakchott 2016-2024\n'
             'Gris clair = VV brut mensuel  |  Couleur = moyenne mobile 12 mois  |  '
             'Zone bleue = saison des pluies (Jul-Oct)',
             fontsize=13, fontweight='bold')

for ax, zone in zip(axes.flat, ZONE_ORDER):
    sub   = (df_monthly[df_monthly['zone']==zone]
             .sort_values(['year','month'])
             .dropna(subset=['vv_mean'])
             .copy())
    sub['date']   = pd.to_datetime({'year':sub['year'],'month':sub['month'],'day':15})
    sub['mm12']   = sub['vv_mean'].rolling(window=12, center=True, min_periods=6).mean()
    color         = ZONE_COLORS[zone]

    # Zones pluies (juil-oct) sur toutes les années
    for year in YEARS_MONTHLY:
        if date(year, 7, 1) > MONTHLY_LAST:      # pas de saison des pluies 2026
            continue
        ax.axvspan(pd.Timestamp(f'{year}-07-01'), pd.Timestamp(f'{year}-10-31'),
                   alpha=0.06, color='#457b9d', linewidth=0)

    # VV brut mensuel
    ax.plot(sub['date'], sub['vv_mean'], color='#cccccc', lw=0.8, alpha=0.7, zorder=1)

    # Moyenne mobile 12 mois
    ax.plot(sub['date'], sub['mm12'], color=color, lw=2.5, zorder=3,
            label='MM-12')

    # Tendance STL si disponible
    if zone in stl_results:
        full_idx = pd.date_range(f'{YEAR_START}-01-01', f'{MONTHLY_END}-01', freq='MS')
        ax.plot(full_idx, stl_results[zone].trend,
                color=color, lw=1.5, ls='--', alpha=0.6, zorder=2, label='Tendance STL')

    # Annotation amplitude et tendance
    if zone in kw_results:
        amp  = kw_results[zone]['amp']
        p_kw = kw_results[zone]['p']
        sig  = '✓ sig.' if p_kw<0.05 else 'ns'
        ax.text(0.02, 0.05,
                f'Amp. sais. = {amp:.2f} dB ({sig})',
                transform=ax.transAxes, fontsize=7.5, color=color,
                bbox=dict(facecolor='white', alpha=0.75, boxstyle='round', linewidth=0))

    ax.set_title(zone, fontsize=10, fontweight='bold', color=color)
    ax.set_ylabel('VV (dB)', fontsize=8)
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(alpha=0.2, ls=':')
    ax.xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y'))
    ax.tick_params(axis='x', labelsize=7, rotation=45)

plt.tight_layout()
plt.savefig('../outputs/monthly_06_final_synthesis.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: monthly_06_final_synthesis.png')

# ── Figure ville globale ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 6))
fig.suptitle('Indice VV Mensuel — Ville de Nouakchott (pondéré superficie) 2016-2024\n'
             'Gris = brut mensuel  |  Bleu foncé = MM-12  |  Rouge = tendance STL',
             fontsize=13, fontweight='bold')

df_city_monthly_sorted = df_city_monthly.sort_values(['year','month']).copy()
df_city_monthly_sorted['date'] = pd.to_datetime(
    {'year':df_city_monthly_sorted['year'],
     'month':df_city_monthly_sorted['month'],'day':15})
df_city_monthly_sorted['mm12'] = (df_city_monthly_sorted['vv_city']
                                   .rolling(window=12, center=True, min_periods=6).mean())

for year in YEARS_MONTHLY:
    ax.axvspan(pd.Timestamp(f'{year}-07-01'), pd.Timestamp(f'{year}-10-31'),
               alpha=0.08, color='#457b9d', linewidth=0)

ax.plot(df_city_monthly_sorted['date'], df_city_monthly_sorted['vv_city'],
        color='#cccccc', lw=0.8, alpha=0.8, label='VV brut mensuel')
ax.plot(df_city_monthly_sorted['date'], df_city_monthly_sorted['mm12'],
        color='#1d3557', lw=3, label='Moyenne mobile 12 mois')

# STL ville
sub_city   = df_city_monthly_sorted.dropna(subset=['vv_city'])
full_idx_c = pd.date_range(f'{YEARS_MONTHLY[0]}-01-01',
                            f'{YEARS_MONTHLY[-1]}-12-01', freq='MS')
ts_city    = sub_city.set_index('date')['vv_city'].reindex(full_idx_c).interpolate()
stl_city   = STL(ts_city, period=12, robust=True).fit()
ax.plot(full_idx_c, stl_city.trend, color='#e63946', lw=2, ls='--',
        alpha=0.9, label='Tendance STL')

slope_city = np.polyfit(range(len(stl_city.trend)), stl_city.trend, 1)[0] * 12
ax.set_ylabel('VV moyen pondéré (dB)', fontsize=11)
ax.set_xlabel('Date', fontsize=11)
ax.legend(fontsize=10, loc='upper left')
ax.grid(alpha=0.25, ls=':')
ax.xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y'))
ax.tick_params(axis='x', labelsize=9)

ax.text(0.98, 0.05,
        f'Tendance STL ville : {slope_city:+.3f} dB/an',
        transform=ax.transAxes, ha='right', fontsize=10, color='#e63946',
        fontweight='bold',
        bbox=dict(facecolor='white', alpha=0.8, boxstyle='round', linewidth=0))

plt.tight_layout()
plt.savefig('../outputs/monthly_06b_city_global_monthly.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: monthly_06b_city_global_monthly.png')
print()
print('=== RÉCAPITULATIF PARTIE 11 ===')
print(f'  Fichiers produits :')
for f in ['monthly_01_heatmap.png', 'monthly_01b_heatmap_city.png',
          'monthly_02_maps_seasonal.png', 'monthly_03_boxplot_seasonality.png',
          'monthly_03b_seasonal_profile_city.png', 'monthly_04_stl_decomposition.png',
          'monthly_04b_kruskal_wallis.png', 'monthly_05_map_diff_saisons.png',
          'monthly_06_final_synthesis.png', 'monthly_06b_city_global_monthly.png',
          'monthly_sar_vv.csv']:
    print(f'  ✓ /outputs/{f}')

---
## 📊 PARTIE 12 — Indice Trimestriel Désaisonnalisé (Option C — STL Trend)

### Principe
On réutilise la **composante tendance STL** déjà calculée en Partie 11.  
La tendance STL est par définition **désaisonnalisée** — elle ne contient plus le cycle annuel.  
On la rééchantillonne à fréquence **trimestrielle** pour obtenir ~36 points par zone.

### Avantages vs indice annuel
| Critère | Annuel | Trimestriel STL |
|---------|--------|----------------|
| Points par zone | 9 | ~36 |
| Saisonnalité | Non contrôlée | ✓ Éliminée par STL |
| Stabilité | ✓ Bonne | ✓ Bonne (lissage STL) |
| Tests statistiques | Limités (n=9) | Plus puissants (n=36) |
| Comparabilité inter-zones | ✓ | ✓ |

### Workflow
1. Extraire tendance STL → rééchantillonner Q1/Q2/Q3/Q4
2. Convertir en fraction urbaine % via seuil -14 dB (calibration sur l'annuel)
3. Visualiser : séries trimestrielles + heatmap + carte choroplèthe animée
4. Tests : Mann-Kendall trimestriel, accélération/décélération de croissance
5. Indice ville trimestriel pondéré superficie

### 📥 Cellule A — Extraction Tendance STL Trimestrielle

In [ ]:
# ── CELL 12A — Tendance STL → Rééchantillonnage Trimestriel ────────────────
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import linregress


# SUPPRIMER : YEARS_MONTHLY = list(range(2016, 2025))
full_idx = pd.date_range(f'{YEAR_START}-01-01', f'{MONTHLY_END}-01', freq='MS')

assert len(full_idx) == len(stl_results[ZONE_ORDER[0]].trend), \
    f'full_idx={len(full_idx)} vs STL={len(stl_results[ZONE_ORDER[0]].trend)} — désaligné'

records_q = []
for zone in ZONE_ORDER:
    res = stl_results[zone]
    ts_trend = pd.Series(res.trend, index=full_idx, name='trend_vv')
    ts_q = ts_trend.resample('QS').mean()
    for date, vv_trend in ts_q.items():
        if np.isnan(vv_trend):
            continue
        quarter = (date.month - 1) // 3 + 1
        records_q.append({
            'zone'     : zone,
            'date'     : date,
            'year'     : date.year,
            'quarter'  : quarter,
            'label'    : f'{date.year}-Q{quarter}',
            'vv_trend' : round(float(vv_trend), 4),
        })

df_q = pd.DataFrame(records_q)

# ── Calibration zone-par-zone (régression VV_trend_Q3 ~ UF_annuel) ──────────
# CORRECTION : la calibration globale VV_ZERO=-14→UF=0% était fausse.
# La tendance STL = VV moyen zone (mélange bâti+sable), pas pixel pur.
# Une zone à 50% bâti a VV moyen ≈ -14 dB → mappée à 0% avec l'ancienne formule.
# Fix : régression linéaire par zone calée sur les UF annuelles connues.

UF_ANNUAL = {z: df_ts.loc[z].values.tolist() for z in ZONE_ORDER}

zone_calib = {}
print('Calibration zone-par-zone (VV_trend_Q3 ~ UF_annuel) :')
print(f'{"Zone":20s} {"a":>8s} {"b":>10s} {"r²":>6s}')
print('-'*48)
for zone in ZONE_ORDER:
    sub_q3 = (df_q[df_q['zone']==zone]
              .query('quarter==3')
              .sort_values('year'))
    vv = sub_q3['vv_trend'].values
    uf = np.array(UF_ANNUAL[zone])
    n  = min(len(vv), len(uf))
    slope, intercept, r, _, _ = linregress(vv[:n], uf[:n])
    zone_calib[zone] = (slope, intercept)
    print(f'{zone:20s} {slope:>8.4f} {intercept:>10.4f} {r**2:>6.3f}')

df_q['uf_trend'] = df_q.apply(
    lambda row: float(np.clip(
        zone_calib[row['zone']][0] * row['vv_trend'] + zone_calib[row['zone']][1],
        0, 100
    )), axis=1
)

# Vérification cohérence
print('\nVérification cohérence UF trimestrielle vs UF annuelle :')
print(f'{"Zone":20s} {"UF_annuel_2024":>15s} {"UF_trim_2024":>14s} {"Écart":>8s}')
print('-'*62)
for zone in ZONE_ORDER:
    sub_q   = df_q[(df_q['zone']==zone) & (df_q['year']==2024)]
    uf_q_24 = sub_q['uf_trend'].mean() if len(sub_q)>0 else np.nan
    uf_a_24 = UF_ANNUAL[zone][-1]
    ecart   = uf_q_24 - uf_a_24 if not np.isnan(uf_q_24) else np.nan
    print(f'{zone:20s} {uf_a_24:>15.1f} {uf_q_24:>14.1f} {ecart:>+8.1f}')

df_q.to_csv('../outputs/quarterly_uf_stl.csv', index=False)
print(f'\n✓ {len(df_q)} enregistrements trimestriels')
print(f'  Zones : {df_q["zone"].nunique()}')
print(f'  Période : {df_q["label"].iloc[0]} → {df_q["label"].iloc[-1]}')
print('✓ Saved: ../outputs/quarterly_uf_stl.csv')

### 📈 Cellule B — Séries Trimestrielles par Moughataa

In [ ]:
# ── CELL 12B — Visualisation Séries Trimestrielles ──────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(22, 16))
fig.suptitle('Indice Urbain Trimestriel Désaisonnalisé — Nouakchott 2016-2024\n'
             'Basé sur la composante Tendance STL  |  Saisonnalité éliminée',
             fontsize=14, fontweight='bold')

for ax, zone in zip(axes.flat, ZONE_ORDER):
    sub   = df_q[df_q['zone']==zone].sort_values('date')
    color = ZONE_COLORS[zone]
    dates = pd.to_datetime(sub['date'])

    ax.plot(dates, sub['uf_trend'], color=color, lw=2.5,
            marker='o', ms=4, zorder=3, label='UF trimestrielle')

    rolling_std = sub['uf_trend'].rolling(4, center=True, min_periods=2).std()
    ax.fill_between(dates,
                    sub['uf_trend'] - rolling_std,
                    sub['uf_trend'] + rolling_std,
                    color=color, alpha=0.12, zorder=1, label='±1σ glissant')

    # Diamants annuels — indice annuel de référence
    # Diamants annuels — indice annuel de référence (YEARS, pas YEARS_MONTHLY !)
    for i, year in enumerate(YEARS):
        ax.scatter(pd.Timestamp(f'{year}-07-01'), UF_ANNUAL[zone][i],
                   marker='D', s=40, color='black', zorder=5, alpha=0.6)

    # Régression
    x_num = np.arange(len(sub))
    slope, intercept, r, p, _ = stats.linregress(x_num, sub['uf_trend'])
    trend_line = intercept + slope * x_num
    ax.plot(dates, trend_line, color=color, lw=1.5, ls='--', alpha=0.7, zorder=2)

    slope_yr = slope * 4
    sig = '✓' if p < 0.05 else 'ns'
    ax.text(0.03, 0.06,
            f'+{slope_yr:.2f} pp/an  (p={p:.3f} {sig})',
            transform=ax.transAxes, fontsize=8, color=color,
            bbox=dict(facecolor='white', alpha=0.75, boxstyle='round', linewidth=0))

    ax.set_title(zone, fontsize=10, fontweight='bold', color=color)
    ax.set_ylabel('Fraction Urbaine (%)', fontsize=8)
    ax.set_ylim(bottom=max(0, sub['uf_trend'].min()-5))
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(alpha=0.2, ls=':')
    ax.xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y'))
    ax.tick_params(axis='x', labelsize=7, rotation=45)

axes[0,0].scatter([], [], marker='D', s=40, color='black', alpha=0.6,
                  label='Indice annuel (référence)')
axes[0,0].legend(fontsize=7, loc='upper left')

plt.tight_layout()
plt.savefig('../outputs/quarterly_01_series.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: quarterly_01_series.png')

### 🗓️ Cellule C — Heatmap Trimestrielle + Détection Accélération

In [ ]:
# ── CELL 12C — Heatmap Trimestrielle et Accélération de Croissance ─────────
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
fig.suptitle('Indice Trimestriel — Heatmap et Accélération de Croissance\n'
             'Nouakchott 2016-2024  |  Désaisonnalisé STL',
             fontsize=13, fontweight='bold')

ax = axes[0]
pivot_q = df_q.pivot_table(index='zone', columns='label',
                            values='uf_trend', aggfunc='mean')
zone_order_uf = sorted(ZONE_ORDER, key=lambda z: UF_ANNUAL[z][-1], reverse=True)
pivot_q = pivot_q.reindex(zone_order_uf)

cmap_hm = LinearSegmentedColormap.from_list('uf_hm',
    ['#f0f4f8','#c9d6e3','#7fb3d3','#2c7bb6','#1a3a5c'])
im = ax.imshow(pivot_q.values, aspect='auto', cmap=cmap_hm,
               vmin=0, vmax=100, interpolation='nearest')

for i in range(len(pivot_q.index)):
    for j in range(len(pivot_q.columns)):
        if j % 2 == 0:
            val = pivot_q.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.0f}', ha='center', va='center',
                        fontsize=5.5,
                        color='white' if val > 50 else 'black')

ax.set_yticks(range(len(zone_order_uf)))
ax.set_yticklabels(zone_order_uf, fontsize=8)
q_labels  = list(pivot_q.columns)
q1_idx    = [i for i, l in enumerate(q_labels) if 'Q1' in l]
q1_labels = [l.replace('-Q1','') for l in q_labels if 'Q1' in l]
ax.set_xticks(q1_idx)
ax.set_xticklabels(q1_labels, fontsize=8, rotation=45)
ax.set_title('Fraction Urbaine % par Zone et Trimestre', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.7, label='UF (%)')

ax2 = axes[1]
periods = [
    ('2016-2018', 2016, 2018, '#457b9d'),
    ('2019-2021', 2019, 2021, '#2a9d8f'),
    ('2022-2024', 2022, 2024, '#e63946'),
]
slopes_by_period = {z: {} for z in ZONE_ORDER}
for label_p, y1, y2, color_p in periods:
    for zone in ZONE_ORDER:
        sub = df_q[(df_q['zone']==zone) &
                   (df_q['year']>=y1) & (df_q['year']<=y2)].sort_values('date')
        if len(sub) < 4:
            continue
        x = np.arange(len(sub))
        slope, _, _, p, _ = stats.linregress(x, sub['uf_trend'])
        slopes_by_period[zone][label_p] = slope * 4

x_pos  = np.arange(len(ZONE_ORDER))
width  = 0.25
for i, (label_p, y1, y2, color_p) in enumerate(periods):
    vals = [slopes_by_period[z].get(label_p, 0) for z in ZONE_ORDER]
    ax2.bar(x_pos + i*width, vals, width, label=label_p,
            color=color_p, alpha=0.85, edgecolor='white')

ax2.axhline(0, color='black', lw=1)
ax2.set_xticks(x_pos + width)
ax2.set_xticklabels(ZONE_ORDER, rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('Vitesse de croissance (pp/an)', fontsize=10)
ax2.set_title('Accélération / Décélération par Période\n'
              '(pente de régression sur sous-période)', fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.25, ls=':')

fastest = max(ZONE_ORDER,
              key=lambda z: slopes_by_period[z].get('2022-2024', -999))
ax2.text(0.98, 0.97,
         f'Croissance la plus rapide\n2022-2024 : {fastest}',
         transform=ax2.transAxes, ha='right', va='top', fontsize=9,
         bbox=dict(facecolor='#e63946', alpha=0.15, boxstyle='round'))

plt.tight_layout()
plt.savefig('../outputs/quarterly_02_heatmap_acceleration.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: quarterly_02_heatmap_acceleration.png')

print()
print('Vitesses de croissance (pp/an) par sous-période :')
print(f'{"Zone":20s} {"2016-18":>10s} {"2019-21":>10s} {"2022-24":>10s} {"Accélération":>14s}')
print('-'*58)
for zone in ZONE_ORDER:
    s1 = slopes_by_period[zone].get('2016-2018', np.nan)
    s2 = slopes_by_period[zone].get('2019-2021', np.nan)
    s3 = slopes_by_period[zone].get('2022-2024', np.nan)
    acc  = s3 - s1 if not np.isnan(s1) and not np.isnan(s3) else np.nan
    flag = '↑ accélère' if acc > 0.3 else ('↓ ralentit' if acc < -0.3 else '→ stable')
    print(f'{zone:20s} {s1:>+10.2f} {s2:>+10.2f} {s3:>+10.2f} {flag}')

### 📐 Cellule D — Test Mann-Kendall Trimestriel + Breakpoints

In [ ]:
# ── CELL 12D — Mann-Kendall (implémentation manuelle — pas de dépendance externe) ──
from scipy.stats import norm as _norm

def mann_kendall_test(x):
    x = np.array(x, dtype=float)
    n = len(x)
    s = sum(np.sign(x[j] - x[k]) for k in range(n-1) for j in range(k+1, n))
    var_s = n*(n-1)*(2*n+5)/18
    z = (s-1)/np.sqrt(var_s) if s>0 else (s+1)/np.sqrt(var_s) if s<0 else 0.0
    p = 2*(1 - _norm.cdf(abs(z)))
    tau = s / (0.5*n*(n-1))
    slopes = [(x[j]-x[k])/(j-k) for k in range(n-1) for j in range(k+1, n)]
    slope  = np.median(slopes)
    trend  = ('increasing' if p<0.05 and s>0
              else 'decreasing' if p<0.05 and s<0
              else 'no trend')
    class Result:
        pass
    r = Result()
    r.Tau=tau; r.p=p; r.s=s; r.slope=slope; r.trend=trend
    return r

print('Test de Mann-Kendall sur les séries trimestrielles désaisonnalisées')
print('H0 : pas de tendance monotone')
print('='*72)
print(f'{"Zone":20s} {"stat Tau":>10s} {"p-value":>10s} {"Tendance":>12s} {"slope":>10s} {"sig":>6s}')
print('-'*72)

mk_results = {}
for zone in ZONE_ORDER:
    sub = df_q[df_q['zone']==zone].sort_values('date')['uf_trend'].dropna()
    res = mann_kendall_test(sub)
    slope_q = res.slope * 4   # trimestre → annuel
    mk_results[zone] = {
        'tau': res.Tau, 'p': res.p, 'trend': res.trend,
        'slope': slope_q, 's': res.s
    }
    sig = ('***' if res.p<0.001 else '**' if res.p<0.01
           else '*' if res.p<0.05 else 'ns')
    print(f'{zone:20s} {res.Tau:>+10.3f} {res.p:>10.4f} '
          f'{res.trend:>12s} {slope_q:>+10.3f} {sig:>6s}')

print()
sig_zones = [z for z in ZONE_ORDER if mk_results[z]['p'] < 0.05]
print(f'  {len(sig_zones)}/9 zones significatives (p<0.05)')
for z in sig_zones:
    print(f'  ✓ {z}: {mk_results[z]["slope"]:+.2f} pp/an (p={mk_results[z]["p"]:.4f})')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Test de Mann-Kendall — Tendance Trimestrielle Désaisonnalisée\n'
             'Nouakchott 2016-2024  |  Toutes zones', fontsize=13, fontweight='bold')

ax = axes[0]
taus  = [mk_results[z]['tau'] for z in ZONE_ORDER]
pvals = [mk_results[z]['p']   for z in ZONE_ORDER]
cols  = [ZONE_COLORS[z] if p<0.05 else '#adb5bd' for z,p in zip(ZONE_ORDER, pvals)]
bars  = ax.barh(ZONE_ORDER, taus, color=cols, alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Tau de Kendall  [+1 = tendance croissante parfaite]')
ax.set_title('Tau de Kendall par Moughataa', fontweight='bold')
for bar, p in zip(bars, pvals):
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else ''
    if sig:
        ax.text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
                sig, va='center', fontsize=10, color='#2a9d8f', fontweight='bold')
ax.grid(axis='x', alpha=0.3, ls=':')

ax2 = axes[1]
slopes = [mk_results[z]['slope'] for z in ZONE_ORDER]
cols2  = [ZONE_COLORS[z] for z in ZONE_ORDER]
ax2.barh(ZONE_ORDER, slopes, color=cols2, alpha=0.85, edgecolor='white')
ax2.axvline(0, color='black', lw=1)
ax2.set_xlabel('Vitesse de croissance urbaine (pp/an)')
ax2.set_title('Pente de Theil-Sen (pp/an)', fontweight='bold')
ax2.grid(axis='x', alpha=0.3, ls=':')

plt.tight_layout()
plt.savefig('../outputs/quarterly_03_mann_kendall.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: quarterly_03_mann_kendall.png')

### 🌍 Cellule E — Indice Ville Trimestriel + Synthèse Finale

In [ ]:
# ── CELL 12E — Indice Ville Trimestriel + Figure Synthèse ───────────────────

city_q_records = []
for label_q in sorted(df_q['label'].unique()):
    sub = df_q[df_q['label']==label_q]
    total_w = 0; weighted_uf = 0
    for zone in ZONE_ORDER:
        row = sub[sub['zone']==zone]
        if len(row)==0: continue
        val = row['uf_trend'].values[0]
        if np.isnan(val): continue
        w = areas[zone]
        weighted_uf += val * w
        total_w     += w
    if total_w > 0:
        date_q = pd.to_datetime(df_q[df_q['label']==label_q]['date'].iloc[0])
        city_q_records.append({
            'label'  : label_q,
            'date'   : date_q,
            'year'   : date_q.year,
            'quarter': (date_q.month-1)//3+1,
            'uf_city': round(weighted_uf/total_w, 3)
        })

df_city_q = pd.DataFrame(city_q_records).sort_values('date')

res_city     = mann_kendall_test(df_city_q['uf_city'].dropna())
slope_city_q = res_city.slope * 4

print(f'Indice ville trimestriel :')
print(f'  Mann-Kendall Tau = {res_city.Tau:+.3f}  p = {res_city.p:.4f}  '
      f'tendance = {res_city.trend}')
print(f'  Pente Theil-Sen  = {slope_city_q:+.3f} pp/an')

fig, axes = plt.subplots(2, 1, figsize=(18, 14),
                          gridspec_kw={'height_ratios': [2, 1]})
fig.suptitle('Indice Urbain Trimestriel Désaisonnalisé — Synthèse Finale\n'
             'Nouakchott 2016-2024  |  Basé sur Tendance STL Sentinel-1 SAR',
             fontsize=14, fontweight='bold')

ax = axes[0]
for zone in ZONE_ORDER:
    sub   = df_q[df_q['zone']==zone].sort_values('date')
    dates = pd.to_datetime(sub['date'])
    color = ZONE_COLORS[zone]
    ax.plot(dates, sub['uf_trend'], color=color, lw=1.8, alpha=0.85, label=zone)
    last = sub.iloc[-1]
    zone_label = f"{zone.split()[0]}\n{last['uf_trend']:.0f}%"
    ax.annotate(zone_label,
                (pd.to_datetime(last['date']), last['uf_trend']),
                xytext=(8, 0), textcoords='offset points',
                fontsize=6.5, color=color, va='center')

ax.set_ylabel('Fraction Urbaine Estimée (%)', fontsize=11)
ax.set_title('Trajectoires Trimestrielles par Moughataa', fontweight='bold')
ax.legend(fontsize=8, loc='upper left', ncol=3)
ax.grid(alpha=0.2, ls=':')
ax.xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y'))
ax.tick_params(axis='x', labelsize=9)
for year in YEARS_MONTHLY:
    ax.axvline(pd.Timestamp(f'{year}-01-01'), color='grey',
               lw=0.5, ls=':', alpha=0.4)

ax2 = axes[1]
ax2.fill_between(df_city_q['date'], df_city_q['uf_city'],
                 alpha=0.15, color='#1d3557')
ax2.plot(df_city_q['date'], df_city_q['uf_city'],
         color='#1d3557', lw=3, marker='o', ms=4, label='Indice ville')

x_city = np.arange(len(df_city_q))
s_c, i_c, _, _, _ = stats.linregress(x_city, df_city_q['uf_city'])
ax2.plot(df_city_q['date'], i_c + s_c*x_city,
         color='#e63946', lw=2, ls='--',
         label=f'Tendance ({s_c*4:+.2f} pp/an)')

for year in YEARS_MONTHLY:
    sub_y = df_city_q[df_city_q['year']==year]
    if len(sub_y) > 0:
        mean_y = sub_y['uf_city'].mean()
        ax2.annotate(f'{mean_y:.1f}%',
                     (pd.Timestamp(f'{year}-07-01'), mean_y),
                     xytext=(0, 8), textcoords='offset points',
                     fontsize=7.5, ha='center', color='#1d3557')

ax2.set_ylabel('Fraction Urbaine Ville (%)', fontsize=11)
ax2.set_title(f'Indice Ville Global — Mann-Kendall : Tau={res_city.Tau:+.3f}  '
              f'p={res_city.p:.4f}  ({res_city.trend})', fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.2, ls=':')
ax2.xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
ax2.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y'))
ax2.tick_params(axis='x', labelsize=9)

plt.tight_layout()
plt.savefig('../outputs/quarterly_04_final_synthesis.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Saved: quarterly_04_final_synthesis.png')

print()
print('=== RÉCAPITULATIF PARTIE 12 ===')
print(f'  Indice trimestriel : {len(df_city_q)} points ville  |  {len(df_q)} points zone')
print(f'  Tendance ville     : {slope_city_q:+.3f} pp/an  (p={res_city.p:.4f}, {res_city.trend})')
for f in ['quarterly_01_series.png', 'quarterly_02_heatmap_acceleration.png',
          'quarterly_03_mann_kendall.png', 'quarterly_04_final_synthesis.png',
          'quarterly_uf_stl.csv']:
    print(f'  ✓ /outputs/{f}')

In [ ]:
# ── CELL 12F — Dataset Économétrique Trimestriel (SAR × ICC) ────────────────
# Le vrai gain du mémoire : n ≈ 41 trimestres au lieu de n = 9 années.
# Prérequis : df_q (12A) · areas (17) · df_icc_q, icc_cols, WILAYA_TO_MOUGHATAA (20)

def q_label(d):
    return f'{d.year}-Q{(d.month - 1)//3 + 1}'

# ── Garde-fou : df_q doit couvrir 42 trimestres (2016-Q1 → 2026-Q2) ────────
_nq = df_q['date'].nunique()
assert _nq == 42, (
    f'df_q ne contient que {_nq} trimestres (fin : {q_label(pd.to_datetime(df_q["date"]).max())}). '
    f'Attendu 42 → 2026-Q2. La CELL 12A n\'a pas été remplacée : '
    f'vérifier que full_idx = pd.date_range(YEAR_START-01-01, MONTHLY_END-01, freq="MS").')

# ── 1. Série ville trimestrielle (moyenne pondérée par superficie) ──────────
_w = pd.Series({z: areas[z] for z in ZONE_ORDER})
_w = _w / _w.sum()

pivot_uf = df_q.pivot_table(index='date', columns='zone', values='uf_trend')
sar_city_q = pivot_uf[list(_w.index)].mul(_w, axis=1).sum(axis=1).rename('sar_uf')

# ── 2. Jointure avec l'ICC trimestriel (inner join → recouvrement commun) ───
df_ts_q = pd.concat([sar_city_q, df_icc_q[icc_cols]], axis=1, join='inner').sort_index()

print('★ DATASET TRIMESTRIEL — série ville')
print(f'  n = {len(df_ts_q)} trimestres  '
      f'[{q_label(df_ts_q.index[0])} → {q_label(df_ts_q.index[-1])}]')
print(f'  SAR seul : {len(sar_city_q)} trim. (→ {q_label(sar_city_q.index[-1])}) — '
      f'l\'indice satellite dépasse l\'ICC : c\'est l\'argument « haute fréquence »')
print()
print(df_ts_q.head(3).round(2).to_string())
print('  ...')
print(df_ts_q.tail(3).round(2).to_string())
df_ts_q.to_csv('../outputs/eco_quarterly_city.csv')

# ── 3. Panel trimestriel : 9 zones × trimestres ────────────────────────────
df_panel_q = (df_q[['zone', 'date', 'uf_trend']]
              .merge(df_icc_q[icc_cols], left_on='date', right_index=True, how='inner')
              .sort_values(['zone', 'date'])
              .reset_index(drop=True))
df_panel_q['label']  = df_panel_q['date'].map(q_label)
df_panel_q['wilaya'] = df_panel_q['zone'].map(
    {m: w for w, ms in WILAYA_TO_MOUGHATAA.items() for m in ms})

assert df_panel_q['wilaya'].notna().all(), 'Zone non mappée à une wilaya — vérifier les noms'

print(f'\n★ PANEL TRIMESTRIEL : {len(df_panel_q)} observations '
      f'({df_panel_q["zone"].nunique()} zones × {df_panel_q["date"].nunique()} trimestres)')
print(df_panel_q.groupby('wilaya')['zone'].unique().to_string())
df_panel_q.to_csv('../outputs/eco_quarterly_panel.csv', index=False)
print('\n✓ Saved: eco_quarterly_city.csv · eco_quarterly_panel.csv')

In [ ]:
# ── CELL 12G — Économétrie Trimestrielle : SAR CVS × ICC  (n ≈ 41) ─────────
from statsmodels.tsa.stattools import adfuller, grangercausalitytests, coint
import warnings; warnings.filterwarnings('ignore')
# ── 0. Dépendances (rend 12G autonome — n'exige plus que 12F ait tourné) ────
def q_label(d):
    return f'{d.year}-Q{(d.month - 1)//3 + 1}'

_w = pd.Series({z: areas[z] for z in ZONE_ORDER}); _w /= _w.sum()

# série ville « tendance lissée » (normalement produite en 12F)
if 'sar_city_q' not in globals():
    sar_city_q = (df_q.pivot_table(index='date', columns='zone', values='uf_trend')[list(_w.index)]
                      .mul(_w, axis=1).sum(axis=1).rename('sar_uf'))
    print('ℹ️  sar_city_q reconstruite localement (CELL 12F non exécutée)')
    
# ── 1. Série CVS : Tendance + Résidu = Observé − Saisonnalité ───────────────
#    (et NON la tendance seule : une série lissée fausse ADF/Granger)
records_sa = []
for zone in ZONE_ORDER:
    res  = stl_results[zone]
    a, b = zone_calib[zone]
    sa_m = pd.Series(np.asarray(res.trend) + np.asarray(res.resid), index=full_idx)
    for date_, vv in sa_m.resample('QS').mean().items():
        if np.isnan(vv): continue
        records_sa.append({'zone':zone, 'date':date_,
                           'uf_sa': float(np.clip(a*vv + b, 0, 100))})
df_sa = pd.DataFrame(records_sa)

_w  = pd.Series({z: areas[z] for z in ZONE_ORDER}); _w /= _w.sum()
sar_sa_q = (df_sa.pivot_table(index='date', columns='zone', values='uf_sa')[list(_w.index)]
                 .mul(_w, axis=1).sum(axis=1).rename('sar_sa'))

# ── 2. Jointure avec l'ICC ─────────────────────────────────────────────────
dfq = pd.concat([sar_sa_q, sar_city_q.rename('sar_trend'), df_icc_q[icc_cols]],
                axis=1, join='inner').sort_index().dropna(subset=['sar_sa','icc_global'])
N = len(dfq)
print(f'★ ÉCONOMÉTRIE TRIMESTRIELLE — n = {N} trimestres '
      f'[{q_label(dfq.index[0])} → {q_label(dfq.index[-1])}]\n')

# ── 3. ADF : niveaux puis différences, sur les DEUX séries SAR ─────────────
print('ÉTAPE 1 — ADF (H0 : racine unitaire)')
print('='*68)
adf_q = {}
for nm, s in [('SAR CVS (trend+resid)', dfq['sar_sa']),
              ('SAR lissé (trend seul)', dfq['sar_trend']),
              ('ICC global',             dfq['icc_global']),
              ('ICC main-œuvre',         dfq['icc_mainoeuvre'])]:
    st_l, p_l = adfuller(s.dropna(), autolag='AIC')[:2]
    st_d, p_d_ = adfuller(s.diff().dropna(), autolag='AIC')[:2]
    adf_q[nm] = (p_l, p_d_)
    print(f'  {nm:24s} niveau p={p_l:.4f} {"NON-STAT" if p_l>0.05 else "STAT ✓":9s} | '
          f'Δ p={p_d_:.4f} {"STAT ✓ → I(1)" if p_d_<0.05 else "non-stat"}')
print('\n  → Le « SAR lissé » paraît toujours plus persistant : c\'est l\'effet du filtre Loess.')
print('    On retient la série CVS pour l\'inférence.')

# ── 4. Corrélation : niveaux (fallacieuse) vs différences (correcte) ───────
print('\nÉTAPE 2 — CORRÉLATION')
print('='*68)
r_lvl, p_lvl = stats.pearsonr(dfq['sar_sa'], dfq['icc_global'])
d1 = dfq[['sar_sa','icc_global']].diff().dropna()
r_dif, p_dif = stats.pearsonr(d1['sar_sa'], d1['icc_global'])
print(f'  Niveaux      : r={r_lvl:+.3f}  p={p_lvl:.4f}   ⚠️ deux séries tendancielles → fallacieux')
print(f'  Différences  : r={r_dif:+.3f}  p={p_dif:.4f}   ← la vraie corrélation')

# ── 5. Cointégration Engle-Granger (n≈41 → puissance correcte) ─────────────
print('\nÉTAPE 3 — COINTÉGRATION ENGLE-GRANGER')
print('='*68)
sc, p_ci, _ = coint(dfq['sar_sa'], dfq['icc_global'])
print(f'  stat={sc:.4f}  p={p_ci:.4f}  → {"COINTÉGRÉES ✓" if p_ci<0.05 else "NON cointégrées"}')
print(f'  (n={N} → contrairement au test annuel n=10, ce résultat est interprétable)')

# ── 6. Causalité de Granger — LES DEUX SENS, sur séries différenciées ──────
print('\nÉTAPE 4 — CAUSALITÉ DE GRANGER (sur Δ, jusqu\'à 4 trimestres)')
print('='*68)
gc_q = {}
for sens, cause, effet in [('ICC → SAR','icc_global','sar_sa'),
                           ('SAR → ICC','sar_sa','icc_global')]:
    arr = d1[[effet, cause]].values          # statsmodels : col2 cause-t-elle col1 ?
    print(f'  {sens} :')
    for lag, res in grangercausalitytests(arr, maxlag=4, verbose=False).items():
        p = res[0]['ssr_ftest'][1]
        gc_q[(sens, lag)] = p
        print(f'     lag {lag} ({lag*3} mois) : p={p:.4f}  '
              f'{"→ CAUSALITÉ ✓" if p<0.05 else "ns"}')

sig = [k for k,v in gc_q.items() if v < 0.05]
print()
if sig:
    print(f'  ⚠️  {len(sig)} relation(s) significative(s) détectée(s) : {sig}')
    print('     → à interpréter : le test annuel (n=10, série décalée) ne pouvait pas les voir.')
else:
    print('  Aucune causalité détectée dans les deux sens.')
    print('  → Confirme la thèse « facteurs exogènes communs » — MAIS cette fois sur')
    print('    n=41 avec la série ICC corrigée : le résultat est enfin crédible.')

dfq.to_csv('../outputs/eco_quarterly_tests.csv')
print('\n✓ Saved: eco_quarterly_tests.csv')

In [ ]:
# ── CELL 12H — Johansen + VECM : causalité avec correction d'erreur ─────────
from statsmodels.tsa.vector_ar.vecm import (coint_johansen, select_order,
                                            select_coint_rank, VECM)
from statsmodels.tsa.stattools import kpss
import warnings; warnings.filterwarnings('ignore')

data = dfq[['sar_sa', 'icc_global']].dropna()
print(f'VECM sur n = {len(data)} trimestres '
      f'[{q_label(data.index[0])} → {q_label(data.index[-1])}]\n')

# ═══ 0. Confirmation KPSS (H0 = STATIONNAIRE, l'inverse de l'ADF) ═══════════
print('ÉTAPE 0 — CONFIRMATION KPSS')
print('='*68)
for nm, s in [('SAR CVS', data['sar_sa']), ('ICC global', data['icc_global'])]:
    for lbl, x in [('niveau', s), ('Δ', s.diff().dropna())]:
        st, p, *_ = kpss(x, regression='c', nlags='auto')
        print(f'  {nm:12s} {lbl:7s} KPSS={st:.3f}  p={p:.3f}  '
              f'→ {"stationnaire ✓" if p>0.05 else "non-stationnaire"}')

# ═══ 1. Sélection des retards ══════════════════════════════════════════════
print('\nÉTAPE 1 — SÉLECTION DES RETARDS')
print('='*68)
lag_sel = select_order(data, maxlags=8, deterministic='ci')
# AIC sélectionne le retard maximal (8) sur n=41 → sur-paramétrage ; on retient Hannan-Quinn
K = max(1, int(lag_sel.hqic))
print(f'  k_ar_diff retenu (HQIC) : {K}   [AIC={lag_sel.aic}, BIC={lag_sel.bic}]')

# ═══ 2. Test de Johansen ═══════════════════════════════════════════════════
print('\nÉTAPE 2 — COINTÉGRATION DE JOHANSEN')
print('='*68)
jo = coint_johansen(data, det_order=0, k_ar_diff=K)
print('  Trace :')
print(f'  {"H0":8s} {"trace":>10s} {"90%":>9s} {"95%":>9s} {"99%":>9s}   Décision')
for i in range(len(jo.lr1)):
    t, cv = jo.lr1[i], jo.cvt[i]
    print(f'  r ≤ {i:<4d} {t:>10.3f} {cv[0]:>9.2f} {cv[1]:>9.2f} {cv[2]:>9.2f}   '
          f'{"REJET H0 ✓" if t>cv[1] else "non rejet"}')
print('  Valeur propre maximale :')
for i in range(len(jo.lr2)):
    t, cv = jo.lr2[i], jo.cvm[i]
    print(f'  r = {i:<4d} {t:>10.3f} {cv[0]:>9.2f} {cv[1]:>9.2f} {cv[2]:>9.2f}   '
          f'{"REJET H0 ✓" if t>cv[1] else "non rejet"}')
r = select_coint_rank(data, det_order=0, k_ar_diff=K, signif=0.05).rank
print(f'\n  → Rang de cointégration : r = {r}')
if r == 0:
    print('  ⚠️  Johansen CONTREDIT Engle-Granger (p=0.0093) → cointégration FRAGILE.')
elif r == 1:
    print('  ✓ Johansen CONFIRME Engle-Granger : 1 relation de long terme.')

# ═══ 3. VECM ═══════════════════════════════════════════════════════════════
if r >= 1:
    print('\nÉTAPE 3 — ESTIMATION DU VECM')
    print('='*68)
    vres = VECM(data, k_ar_diff=K, coint_rank=1, deterministic='ci').fit()
    beta = vres.beta[:, 0] / vres.beta[0, 0]
    print(f'  Long terme (normalisé) : sar_sa = {-beta[1]:+.4f} × icc_global + const.')
    print(f'\n  Coefficients d\'ajustement α (correction d\'erreur) :')
    print(f'  {"Équation":14s} {"α":>10s} {"p-value":>10s}   Interprétation')
    for i, nm in enumerate(['Δ sar_sa', 'Δ icc_global']):
        a, pa = vres.alpha[i, 0], vres.pvalues_alpha[i, 0]
        if pa < 0.05 and a < 0:  interp = "ajuste vers l'équilibre ✓ → causalité LT"
        elif pa < 0.05:          interp = "significatif mais α>0 (⚠ instable)"
        else:                    interp = "n'ajuste pas (faiblement exogène)"
        print(f'  {nm:14s} {a:>+10.4f} {pa:>10.4f}   {interp}')

    print('\nÉTAPE 4 — CAUSALITÉ DE GRANGER AU SEIN DU VECM')
    print('='*68)
    for caused, causing in [('sar_sa', 'icc_global'), ('icc_global', 'sar_sa')]:
        g = vres.test_granger_causality(caused=caused, causing=causing)
        print(f'  {causing} → {caused:12s} stat={g.test_statistic:7.3f}  '
              f'p={g.pvalue:.4f}  {"→ CAUSALITÉ ✓" if g.pvalue<0.05 else "ns"}')
else:
    print('\n  r = 0 → pas de VECM ; on reste sur le VAR en différences (CELL 12G).')